<a href="https://colab.research.google.com/github/jesusvasquez-tec/analitica-empresarial-integrada/blob/main/Semana10/Vasquez_LAB_D10_AEI_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 10 — Pronóstico de demanda

**Semana 10 · TECSUP 2026-II**

**Docente:** PILAR ROCIO SAYAN MEJIA

**Dataset real:** Online Retail II — UCI Machine Learning Repository (ID 502)

**Objetivo:** construir un pronóstico semanal de ventas, seleccionar modelo y política de cantidad con validación temporal sin información futura, evaluarlos una sola vez y decidir si conviene adoptarlos.

**Alumno:** Jesus Vasquez Espinoza

> El notebook termina en una **decisión fundamentada** (adoptar, mantener en evaluación o no adoptar), no solo en el modelo con menor error.

## Trabajo individual y entrega

Esta práctica dirigida es **individual**. Ejecute el notebook en **su propio Google Colab**, verifique los resultados y responda con sus propias interpretaciones. No se requiere informe separado: las respuestas, el cierre ejecutivo y las conclusiones quedan en este notebook.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con los retos resueltos, a la carpeta `semana10/` de su repositorio personal de GitHub.
2. Conserve la versión en su propio Google Colab y compártala con permiso de lectura.
3. En la **actividad de la semana** presente **ambos URLs**: el enlace directo al archivo en GitHub y el enlace a su Colab.

**Identificación**
- Nombre: Jesus Vasquez Espinoza
- Sección: 6C28A
- URL de GitHub:
- URL de Colab: https://colab.research.google.com/drive/1tuQa3NOE3uDllgRMj58nHT7GYJry8BsX?usp=sharing
- Uso de IA declarado: Se utilizó Qwen para refinar la redacción y las interpretaciones

# 1. Pregunta de negocio

**¿Cuántas unidades debería planificar la empresa para las próximas semanas, qué riesgo asume si el pronóstico se equivoca y conviene adoptar el modelo?**

Se distinguen cuatro pasos que no deben confundirse:
1. **Pronóstico de ventas:** estimación de las ventas semanales.
2. **Política de cantidad:** regla que convierte el pronóstico en una cantidad objetivo semanal (pronóstico + colchón).
3. **Evaluación económica:** costo de los excedentes y faltantes que produce esa política.
4. **Decisión de adopción:** usar, mantener en evaluación o no usar el modelo, según su resultado frente al baseline.

### Diseño del laboratorio
| Elemento | Definición |
|---|---|
| Unidad temporal | Semana de lunes a domingo, identificada por la fecha del lunes |
| Semanas válidas | Solo semanas completas según la cobertura del archivo |
| Variable objetivo | Unidades vendidas (registradas) por semana de un SKU del Reino Unido |
| Test final | Últimas 8 semanas completas; se evalúa una sola vez |
| Selección del SKU | Criterios calculados solo antes del test |
| Validación | Backtesting rolling-origin con horizonte de 8 semanas dentro del entrenamiento |
| Modelos | Naïve, Naïve estacional (m = 4) y Exponential Smoothing |
| Regla de selección (única) | Candidatos con costo promedio ≤ mínimo × 1,01 (incluye exactamente el 1 %); si el mínimo es 0, solo empatan los costos 0. Desempate: menor MAE, menor desviación del <br>costo, menor complejidad |
| Política de colchón | Multiplicador 0; 0,5; 1 o 1,5 × MAE previo; el MAE previo de cada origen usa solo errores anteriores a ese origen |

## Actividad 1 — Revisión de conceptos

Complete la tabla con sus propias palabras e incluya un ejemplo breve. Si su definición permite responder la pregunta de apoyo, es suficiente; si no, corríjala antes de continuar.

| Concepto | Definición con sus propias palabras y ejemplo | Pregunta de apoyo |
|---|---|---|
| Serie temporal | Conjunto de observaciones registradas en orden cronológico, <br>donde la secuencia contiene la información del patrón; por <br>ejemplo, ventas semanales de un SKU. | ¿Por qué no se pueden barajar sus filas? |
| Semana completa | Periodo de lunes a domingo donde el archivo cubre los siete<br> días sin interrupciones; por ejemplo, la semana del 7 al 13 de <br>diciembre de 2009. | ¿Por qué se excluye la primera semana del archivo? |
| Tendencia y estacionalidad | La tendencia es un cambio persistente del nivel medio a largo <br>plazo, mientras que la estacionalidad es un patrón que se repite<br> en intervalos fijos del calendario. | ¿Basta con que la serie suba y baje para afirmar estacionalidad? |
| Baseline (Naïve y Naïve estacional) | Modelos de referencia simples que repiten la última observación<br> o la del mismo periodo del ciclo anterior, sin parámetros que<br> ajustar. | ¿Qué significa que un modelo complejo no supere al Naïve? |
| Exponential Smoothing | Familia de modelos que pondera más las observaciones recientes<br> para estimar el nivel, la tendencia y la estacionalidad de la serie. | ¿Más componentes garantizan un mejor pronóstico? |
| Partición temporal (train, validación, test) | División cronológica de los datos en pasado (entrenamiento), <br>bloques intermedios (validación para elegir modelo/política) y <br>futuro reservado (test). | ¿Por qué el test se mira una sola vez? |
| Backtesting rolling-origin | Técnica que evalúa el modelo en múltiples bloques consecutivos<br> del pasado, expandiendo la ventana de entrenamiento en cada <br>iteración. | ¿Qué aporta frente a un único holdout? |
| MAE, RMSE y WAPE | Métricas para cuantificar el error: MAE (error absoluto medio), <br>RMSE (raíz del error cuadrático medio, que penaliza errores <br>grandes) y WAPE (error relativo total). | ¿Cuál traduce el error directamente a unidades? |
| Costo asimétrico del error | Situación en la que equivocarse por exceso (sobrestock) o por <br>defecto (quiebre de stock) tiene consecuencias financieras <br>distintas. | ¿Por qué dos errores de 20 unidades pueden costar distinto? |
| Pronóstico, política de cantidad y evaluación económica | El pronóstico estima la demanda, la política le suma un colchón <br>calibrado para definir la cantidad objetivo, y la evaluación mide <br>el costo de los excedentes o faltantes resultantes. | ¿La cantidad objetivo es lo que hay que comprar? |

In [ ]:
%pip install -q ucimlrepo statsmodels plotly scikit-learn openpyxl

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sklearn
import statsmodels

from ucimlrepo import fetch_ucirepo
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# No se ocultan advertencias globalmente: las de cada ajuste se registran y se revisan al final.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "statsmodels": statsmodels.__version__, "scikit-learn": sklearn.__version__}
print("Versiones:", VERSIONES)

Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


# 2. Descarga de datos reales y trazabilidad

Se intenta primero la API oficial de UCI; si falla, el archivo ZIP oficial, del que se leen y concatenan sus **dos hojas**; si ambas fallan, se sube manualmente el archivo descargado desde https://archive.ics.uci.edu/dataset/502. La celda registra fuente, hojas, dimensiones, rango de fechas y versiones.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Con otras versiones de las bibliotecas, los ajustes de Exponential Smoothing pueden producir cifras ligeramente distintas.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

def leer_dos_hojas(fuente_excel):
    hojas = pd.read_excel(fuente_excel, sheet_name=None)  # "Year 2009-2010" y "Year 2010-2011"
    return pd.concat(hojas.values(), ignore_index=True), list(hojas)

df, fuente, hojas_leidas = None, None, None
try:
    retail = fetch_ucirepo(id=502)
    df, fuente, hojas_leidas = retail.data.original.copy(), "API ucimlrepo (UCI ID 502)", "tabla única entregada por la API"
except Exception as exc:
    print("API ucimlrepo no disponible:", type(exc).__name__)

if df is None:
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as respuesta:
            blob = respuesta.read()
        with ZipFile(BytesIO(blob)) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
            df, hojas_leidas = leer_dos_hojas(BytesIO(z.read(nombre)))
        fuente = "ZIP oficial de UCI"
    except Exception as exc:
        print("ZIP oficial no disponible:", type(exc).__name__)

if df is None:
    locales = sorted(f for f in os.listdir(".")
                     if "online" in f.lower() and "retail" in f.lower() and f.lower().endswith((".xlsx", ".csv")))
    if not locales:
        try:
            from google.colab import files
            print("Suba el archivo online_retail_II.xlsx descargado desde UCI.")
            locales = list(files.upload().keys())
        except ImportError:
            raise FileNotFoundError("Coloque online_retail_II.xlsx en la carpeta de trabajo y vuelva a ejecutar.")
    archivo = locales[0]
    if archivo.lower().endswith(".xlsx"):
        df, hojas_leidas = leer_dos_hojas(archivo)
    else:
        df = pd.read_csv(archivo, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str})
        hojas_leidas = "no aplica (CSV)"
    fuente = f"archivo local: {archivo}"

fechas_raw = pd.to_datetime(df[[c for c in df.columns if "date" in str(c).lower()][0]], errors="coerce")
print("Fuente utilizada:", fuente)
print("Hojas:", hojas_leidas)
print("Dimensión:", df.shape)
print("Rango de fechas:", fechas_raw.min(), "→", fechas_raw.max())
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Versiones:", VERSIONES)
display(df.head())

API ucimlrepo no disponible: DatasetNotFoundError
Fuente utilizada: ZIP oficial de UCI
Hojas: ['Year 2009-2010', 'Year 2010-2011']
Dimensión: (1067371, 8)
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Ejecución: 2026-10-23T06:12:23.251384+00:00
Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Interpretación de resultados**

La ejecución confirma la carga exitosa del dataset Online Retail II desde el archivo ZIP oficial de UCI, garantizando la integridad de los datos al concatenar correctamente las hojas de los periodos 2009-2010 y 2010-2011. El conjunto resultante contiene 1 067 371 transacciones con 8 variables, abarcando el rango temporal esperado desde el 1 de diciembre de 2009 hasta el 9 de diciembre de 2011. La visualización de las primeras filas valida la presencia de las columnas críticas para el análisis (como Invoice, StockCode, Quantity, InvoiceDate y Country), estableciendo la base estructural necesaria para aplicar los filtros de calidad posteriores, como la exclusión de facturas de cancelación y la segmentación exclusiva de ventas en el Reino Unido.


In [ ]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_", ""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["Customer ID", "CustomerID"]),
    "Country": find_col(df.columns, ["Country"]),
}
faltan = [k for k, v in mapping.items() if v is None and k not in ("Description", "CustomerID")]
if faltan:
    raise ValueError(f"No se identificaron columnas obligatorias: {faltan}")
df = df.rename(columns={v: k for k, v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


**Interpretación de resultados**

El diccionario mostrado confirma la normalización exitosa de los nombres de las columnas, mapeando posibles variaciones del conjunto de datos original (como "Invoice" a "InvoiceNo" o "Price" a "UnitPrice") hacia un esquema estandarizado. Este paso garantiza la consistencia estructural de los datos y asegura que los procesos posteriores de control de calidad y agregación puedan acceder de manera fiable a las ocho variables esenciales, evitando errores de clave derivados de las ligeras diferencias de nomenclatura que puede presentar la fuente oficial de UCI.

# 3. Control de calidad

Los `StockCode` nulos se descartan **antes** de convertir a texto; de lo contrario, la ausencia se convertiría en un producto llamado «nan».

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
print("StockCode nulos descartados:", int(df["StockCode"].isna().sum()))
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df = df[df["StockCode"] != ""]
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

print("Fecha mínima:", df["InvoiceDate"].min(), "·", df["InvoiceDate"].min().day_name())
print("Fecha máxima:", df["InvoiceDate"].max(), "·", df["InvoiceDate"].max().day_name())
print("Cancelaciones:", int(cancel.sum()))
print("Quantity <= 0:", int((df["Quantity"] <= 0).sum()))
print("UnitPrice <= 0:", int((df["UnitPrice"] <= 0).sum()))
print("Fechas nulas:", int(df["InvoiceDate"].isna().sum()))

clean = df.loc[(~cancel) & df["InvoiceDate"].notna() & (df["Quantity"] > 0) &
               (df["UnitPrice"] > 0) & df["Country"].notna()].copy()
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
print("Filas válidas UK:", len(uk), "· SKUs UK:", uk["StockCode"].nunique())

StockCode nulos descartados: 0
Fecha mínima: 2009-12-01 07:45:00 · Tuesday
Fecha máxima: 2011-12-09 12:50:00 · Friday
Cancelaciones: 19494
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Fechas nulas: 0
Filas válidas UK: 958501 · SKUs UK: 4905


**Interpretación de resultados**

El output confirma la ejecución exitosa de las reglas de control de calidad sobre el dataset original. Se verifica que el rango temporal abarca desde el 1 de diciembre de 2009 hasta el 9 de diciembre de 2011, y que se identificaron y excluyeron correctamente 19 494 transacciones de cancelación, junto con más de 22 000 registros con cantidades o precios unitarios no positivos. Como resultado de este filtrado, el análisis continúa con un conjunto limpio de 958 501 transacciones válidas correspondientes a 4 905 SKUs únicos del Reino Unido, garantizando que la serie temporal de demanda que se construirá a continuación refleje únicamente ventas reales y no ajustes contables, devoluciones o errores de registro.


### Pregunta 1
¿Por qué no sería correcto sumar directamente `Quantity` sin eliminar cancelaciones y cantidades no positivas?

**Respuesta:**

No sería correcto porque las facturas que comienzan con "C" representan devoluciones o anulaciones, no ventas reales, y las cantidades no positivas corresponden a ajustes contables, regalos o correcciones administrativas. Incluir estos registros distorsionaría la serie temporal al subestimar artificialmente la demanda neta real, creando un patrón de consumo que no refleja el comportamiento genuino del mercado. Dado que un modelo de pronóstico no puede corregir una serie mal construida, es fundamental filtrar estas observaciones para garantizar que la variable objetivo mida exclusivamente las unidades efectivamente vendidas.



# 4. Semanas completas y partición temporal

Las semanas van de **lunes a domingo** y se identifican por la fecha del lunes. Una semana es **completa** solo si el archivo cubre sus siete días: se excluyen la primera semana si el archivo no empieza en lunes y la última si no termina en domingo. Las **últimas 8 semanas completas** se reservan como test final.

Algunas semanas no tienen **ninguna venta registrada** en los datos filtrados. El cierre de la tienda es una **posible explicación no verificada**: los datos no lo confirman.

In [ ]:
def lunes(fechas):
    fechas = pd.to_datetime(fechas)
    return fechas.dt.normalize() - pd.to_timedelta(fechas.dt.weekday, unit="D")

uk["week"] = lunes(uk["InvoiceDate"])
inicio_archivo, fin_archivo = df["InvoiceDate"].min(), df["InvoiceDate"].max()
lunes_inicio = inicio_archivo.normalize() - pd.Timedelta(days=inicio_archivo.weekday())
lunes_fin = fin_archivo.normalize() - pd.Timedelta(days=fin_archivo.weekday())
primera_completa = lunes_inicio if inicio_archivo.weekday() == 0 else lunes_inicio + pd.Timedelta(weeks=1)
ultima_completa = lunes_fin if fin_archivo.weekday() == 6 else lunes_fin - pd.Timedelta(weeks=1)

semanas = pd.date_range(primera_completa, ultima_completa, freq="W-MON")
TEST_H = 8
semanas_test = semanas[-TEST_H:]
semanas_train = semanas[:-TEST_H]

registros = uk.groupby("week").size().reindex(semanas, fill_value=0)
semanas_sin_registro = registros[registros == 0].index

print("Semanas excluidas por incompletas:", lunes_inicio.date(), "y", lunes_fin.date())
print("Semanas completas:", len(semanas), "·", semanas[0].date(), "→", semanas[-1].date())
print("Entrenamiento:", len(semanas_train), "semanas ·", semanas_train[0].date(), "→", semanas_train[-1].date())
print("Test final:", len(semanas_test), "semanas ·", semanas_test[0].date(), "→", semanas_test[-1].date())
print("Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada):",
      [d.date() for d in semanas_sin_registro])

Semanas excluidas por incompletas: 2009-11-30 y 2011-12-05
Semanas completas: 104 · 2009-12-07 → 2011-11-28
Entrenamiento: 96 semanas · 2009-12-07 → 2011-10-03
Test final: 8 semanas · 2011-10-10 → 2011-11-28
Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada): [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]


**Interpretación de resultados**

El output confirma la correcta construcción y partición temporal de la serie, excluyendo las semanas del 30 de noviembre de 2009 y del 5 de diciembre de 2011 por no cubrir el ciclo completo de lunes a domingo, lo cual evita sesgos artificiales en la demanda. El dataset válido queda constituido por 104 semanas completas, las cuales se dividen estricta y cronológicamente en 96 semanas para el conjunto de entrenamiento y 8 semanas reservadas exclusivamente para el test final, garantizando así una evaluación honesta sin fuga de información. Adicionalmente, se identifican dos semanas con cero ventas registradas a finales de diciembre de 2009 y 2010, las cuales se mantienen en la serie como un supuesto de ventas observadas, atribuyéndose de manera plausible, aunque no verificada, a un posible cierre temporal de la tienda durante las festividades.

### Pregunta 2
¿Por qué una partición aleatoria con `train_test_split` sería incorrecta en este problema?

**Respuesta:**

Una partición aleatoria con train_test_split sería incorrecta porque destruiría la dependencia temporal inherente a las series de tiempo, generando una fuga de información (data leakage) que invalidaría la evaluación del modelo. Al mezclar semanas del futuro con las del pasado, el modelo aprendería patrones de periodos posteriores para "pronosticar" periodos anteriores, produciendo un error artificialmente bajo que no se reproduciría en la práctica. En el contexto del laboratorio, donde se requiere estimar la demanda de las próximas 8 semanas utilizando únicamente la historia disponible hasta ese momento, la partición debe respetar estrictamente el orden cronológico: las 96 semanas iniciales para entrenamiento y las 8 semanas finales como test, garantizando así que la evaluación refleje fielmente el desempeño real del modelo al momento de tomar decisiones de inventario.


# 5. Selección reproducible del producto (solo con el entrenamiento)

Criterios, calculados **exclusivamente con las semanas de entrenamiento**:
1. **Periodo con ventas registradas cercano al entrenamiento completo:** primera venta dentro de las **4 primeras** semanas y última venta dentro de las **4 últimas**. El criterio tolera hasta 3 semanas iniciales o finales sin ventas, que luego se completan con cero aunque podrían corresponder a un periodo sin disponibilidad. **No verifica inventario.**
2. **Cobertura:** ventas registradas en al menos el 90 % de las semanas de entrenamiento con algún registro en la tienda.
3. Entre los elegibles, el de **mayor cantidad de unidades**; desempate por más semanas con ventas.

Se comprueba que haya al menos un producto elegible (y al menos dos para el Reto 3).

In [ ]:
uk_train = uk[uk["week"].isin(semanas_train)]
semanas_con_registro_train = [w for w in semanas_train if w not in set(semanas_sin_registro)]

cobertura = (uk_train.groupby(["StockCode", "week"])["Quantity"].sum().reset_index()
             .groupby("StockCode")
             .agg(semanas_con_venta=("week", "nunique"), unidades=("Quantity", "sum"),
                  primera_venta=("week", "min"), ultima_venta=("week", "max"))
             .reset_index())
cobertura["cobertura"] = cobertura["semanas_con_venta"] / len(semanas_con_registro_train)
elegibles = cobertura[
    (cobertura["primera_venta"] <= semanas_train[3]) &      # dentro de las 4 primeras semanas
    (cobertura["ultima_venta"] >= semanas_train[-4]) &      # dentro de las 4 últimas semanas
    (cobertura["cobertura"] >= 0.90)
].sort_values(["unidades", "semanas_con_venta"], ascending=False).reset_index(drop=True)

if len(elegibles) == 0:
    raise ValueError("Ningún SKU cumple los criterios: revise la carga y la limpieza antes de continuar.")
print("SKUs elegibles:", len(elegibles), "· suficientes para el Reto 3:", len(elegibles) >= 2)
SKU = elegibles.loc[0, "StockCode"]
desc = uk.loc[uk["StockCode"] == SKU, "Description"].dropna().astype(str).mode().iloc[0]
print("SKU seleccionado:", SKU, "·", desc)
display(elegibles.head(5))

SKUs elegibles: 389 · suficientes para el Reto 3: True
SKU seleccionado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
0,84077,93,88273,2009-12-07,2011-10-03,0.989362
1,85123A,94,80903,2009-12-07,2011-10-03,1.000000
2,85099B,94,79134,2009-12-07,2011-10-03,1.000000
3,21212,94,67878,2009-12-07,2011-10-03,1.000000
4,17003,88,67514,2009-12-07,2011-10-03,0.936170


**Interpretación de resultados**

El output confirma la aplicación exitosa de los criterios de selección del producto, identificando 389 SKUs elegibles y validando así la disponibilidad de datos suficientes para el desarrollo del Reto 3. De este conjunto, se seleccionó el SKU 84077 ("WORLD WAR 2 GLIDERS ASSTD DESIGNS") por cumplir rigurosamente con las condiciones de cobertura temporal (ventas desde la primera hasta la última semana permitida y una cobertura del 98,94 % de las semanas con registros) y por registrar el mayor volumen de unidades vendidas (88 273) durante el periodo de entrenamiento. Este filtrado garantiza que el análisis de pronóstico se realice sobre una serie con un comportamiento estable y representativo, evitando sesgos derivados de productos con historiales de ventas incompletos o intermitentes.

# 6. Serie semanal de ventas registradas

**Qué representa un cero.** Un cero indica que **no hay ventas registradas** del SKU en esa semana dentro de los datos filtrados. Los datos no permiten distinguir entre:
- **disponibilidad de inventario** con demanda nula;
- **demanda no satisfecha** por falta de inventario (no queda registrada);
- **ausencia de registros** (por ejemplo, semanas sin ninguna venta registrada en la tienda).

Completar con cero es un **supuesto sobre las ventas observadas**: no demuestra demanda cero ni disponibilidad continua. Por eso el laboratorio pronostica **ventas registradas**, que solo aproximan la demanda.

In [ ]:
serie = (uk[uk["StockCode"] == SKU].groupby("week")["Quantity"].sum()
         .reindex(semanas, fill_value=0).astype(float))
serie.name = "unidades"
train = serie.loc[semanas_train]
test = serie.loc[semanas_test]

ceros = serie[serie == 0].index
print("Semanas de la serie:", len(serie), "· semanas sin ventas registradas del SKU:", len(ceros))
print("  coinciden con semanas sin ningún registro en la tienda:", [d.date() for d in ceros if d in set(semanas_sin_registro)])
print("  semanas con registros en la tienda pero sin ventas del SKU:", [d.date() for d in ceros if d not in set(semanas_sin_registro)])
print("Unidades en entrenamiento:", int(train.sum()))

Semanas de la serie: 104 · semanas sin ventas registradas del SKU: 3
  coinciden con semanas sin ningún registro en la tienda: [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]
  semanas con registros en la tienda pero sin ventas del SKU: [datetime.date(2011, 1, 17)]
Unidades en entrenamiento: 88273


**Interpretación de resultados**

El output confirma la construcción exitosa de la serie temporal de 104 semanas para el SKU seleccionado, sustentada por un volumen sólido de 88 273 unidades vendidas durante el periodo de entrenamiento. Se identifican únicamente 3 semanas con cero ventas registradas: dos de ellas (finales de diciembre de 2009 y 2010) coinciden con periodos de inactividad general de la tienda, lo que sugiere un posible cierre temporal por festividades, mientras que la tercera (17 de enero de 2011) representa una semana con actividad operativa en la tienda pero sin ventas de este producto específico. De acuerdo con las reglas del laboratorio, estos ceros se mantienen en la serie estrictamente como un supuesto de "ventas observadas nulas", sin asumir que la demanda real fue cero, ya que podrían corresponder a quiebres de stock no registrados o falta temporal de disponibilidad, garantizando así la integridad cronológica del análisis sin introducir sesgos de interpolación.

# 7. Exploración (solo con el entrenamiento)

La exploración que orienta decisiones usa únicamente el entrenamiento: mirar el test antes de elegir el modelo sería una fuga de información. En este laboratorio las medias móviles de las **últimas 4 y 8 semanas** se usan para **explorar** el nivel; no se evalúan como método de pronóstico (podrían usarse como tal, pero no forman parte de esta comparación).

In [ ]:
explora = pd.DataFrame({"ventas": train})
explora["media_movil_4"] = explora["ventas"].rolling(4).mean()
explora["media_movil_8"] = explora["ventas"].rolling(8).mean()
fig = go.Figure()
for col, nombre in [("ventas", "Ventas registradas"), ("media_movil_4", "Media móvil 4"), ("media_movil_8", "Media móvil 8")]:
    fig.add_trace(go.Scatter(x=explora.index, y=explora[col], name=nombre))
fig.update_layout(title=f"Entrenamiento · SKU {SKU} · {desc}", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

perfil = (explora.assign(semana_anio=explora.index.isocalendar().week.astype(int))
          .groupby("semana_anio")["ventas"].mean().reset_index())
fig = px.line(perfil, x="semana_anio", y="ventas", markers=True,
              title="Perfil promedio por semana del año (entrenamiento)",
              labels={"semana_anio": "Semana del año", "ventas": "Ventas promedio"})
fig.show()

anual = train.groupby(train.index.year).agg(["count", "mean", "median", "max"])
display(anual.round(1))

,count,mean,median,max
2009,4,493.0,384.0,1204.0
2010,52,1034.4,608.0,7063.0
2011,40,812.8,558.0,3216.0


**Interpretación de resultados**

Los tres elementos confirman que la serie del SKU 84077 presenta un comportamiento altamente volátil, dominado por picos puntuales más que por una tendencia sostenida. En el gráfico de la serie temporal, las ventas registradas oscilan frecuentemente entre 0 y 1500 unidades, con eventos extraordinarios que superan las 3000 y 5000 unidades, particularmente entre octubre y diciembre de 2010, donde se alcanza el máximo histórico de aproximadamente 7000 unidades. Las medias móviles de 4 y 8 semanas suavizan esta volatilidad y revelan un nivel base relativamente estable entre 500 y 1000 unidades, con un ligero incremento del nivel medio al pasar de 2009 a 2010, que se atenúa en 2011, lo que sugiere más un comportamiento variable que una tendencia lineal clara.

El perfil promedio por semana del año evidencia un patrón recurrente en las semanas 44 a 48, con picos que superan las 5000 unidades, lo cual es consistente con un comportamiento estacional de fin de año; sin embargo, al contar con menos de dos ciclos anuales completos en el entrenamiento, esta estacionalidad debe considerarse una hipótesis visual que requiere validación estadística adicional. La tabla resumen anual respalda esta lectura: la media de 2010 (£1034.4) es mayor que la de 2011 (£812.8), pero este contraste está fuertemente distorsionado por los picos extremos (máximo de 7063 en 2010 frente a 3216 en 2011), mientras que las medianas (384, 608 y 558) muestran un nivel central mucho más estable y representativo del comportamiento habitual del producto. En conjunto, la evidencia sugiere que los picos observados son eventos puntuales y no quiebres estructurales, dado que la serie retorna a su nivel base después de cada evento.

### Pregunta 3
Identifique evidencia de tendencia, estacionalidad o quiebres en el entrenamiento. Diferencie lo que observa de lo que todavía necesita comprobar, y explique qué representa un cero en la serie.

**Respuesta:**

En el periodo de entrenamiento se observa una posible estacionalidad de fin de año (con picos recurrentes entre las semanas 44 y 50) y una alta volatilidad, evidenciada por el aumento del promedio semanal de 2009 a 2010 y una leve caída en 2011, lo que sugiere más un comportamiento variable que una tendencia lineal clara. Sin embargo, al contar con menos de dos años completos de datos, esta estacionalidad es solo una hipótesis visual que requiere validación estadística (como descomposición de la serie o análisis de autocorrelación), y los picos de ventas extremos (como el máximo de más de 7000 unidades en 2010) se comportan como eventos puntuales y no como quiebres estructurales, dado que la serie retorna a su nivel base posteriormente. Finalmente, un cero en la serie representa estrictamente la "ausencia de ventas registradas" en esa semana; no implica necesariamente una demanda nula, ya que podría deberse a quiebres de stock (demanda censurada) o a cierres temporales de la tienda, tal como se identifica en las semanas donde no hubo ningún registro de actividad en el sistema.


# 8. Modelos, cantidad operativa, costo y regla de selección

- **Naïve:** repite la última semana observada.
- **Naïve estacional (m = 4):** repite la semana de hace 4 semanas. El periodo de 4 semanas es un **supuesto de modelado** (ciclo operativo corto), no una propiedad comprobada del negocio; con 96 semanas de entrenamiento no hay suficientes años para estimar con confianza un ciclo anual de 52 semanas.
- **Exponential Smoothing:** nivel y tendencia aditivos con estacionalidad aditiva m = 4. Si un entrenamiento tuviera menos de 2 × 4 + 8 = 16 semanas, se simplifica a nivel y tendencia (documentado; se informa si ocurre).

**Cantidad operativa:** todo pronóstico se convierte en `max(redondeo(pronóstico), 0)` y **todas** las métricas y costos se calculan sobre esa cantidad.

**Costo base (supuesto académico, no proviene de UCI):** £0,40 por unidad excedente y £1,50 por unidad faltante.

**Regla única de selección** (se usa para el modelo, la política de colchón, el Reto 1 y el Reto 3):
1. Candidatos: costo promedio de validación ≤ mínimo × 1,01 (**incluye** exactamente el 1 %; se admite una holgura de 1e-9 por redondeo numérico). Si el mínimo es 0, solo son candidatos los costos iguales a 0.
2. Desempate entre candidatos: menor MAE promedio, luego menor desviación del costo y, al final, menor complejidad.

In [ ]:
SEASON = 4
OVER_COST, SHORT_COST = 0.40, 1.50
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]   # en orden de complejidad
TOLERANCIA = 0.01
registro_ajustes = []

def ajustar_es(y, etiqueta):
    """Ajusta Exponential Smoothing y registra todas las advertencias del ajuste."""
    if len(y) >= 2 * SEASON + 8:
        modelo = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=SEASON,
                                      initialization_method="estimated")
        especificacion = "nivel + tendencia + estacionalidad aditivas (m = 4)"
    else:
        modelo = ExponentialSmoothing(y, trend="add", seasonal=None, initialization_method="estimated")
        especificacion = "simplificado: nivel + tendencia"
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        ajuste = modelo.fit(optimized=True)
    registro_ajustes.append({"etapa": etiqueta.split(" · ")[0], "ajuste": etiqueta, "n_train": len(y),
                             "especificacion": especificacion, "advertencias": len(avisos),
                             "tipos": "; ".join(sorted({w.category.__name__ for w in avisos})),
                             "mensajes": " | ".join(sorted({str(w.message)[:110] for w in avisos}))})
    return ajuste

def pronosticar(modelo, y, h, etiqueta=""):
    if modelo == "Naive":
        f = np.repeat(y.iloc[-1], h)
    elif modelo == "SeasonalNaive":
        f = np.resize(y.iloc[-SEASON:].values, h)
    elif modelo == "ExpSmoothing":
        f = np.asarray(ajustar_es(y, etiqueta).forecast(h), dtype=float)
    else:
        raise ValueError(modelo)
    return np.maximum(np.round(f), 0)   # cantidad operativa

def evaluar(real, cantidad, over=OVER_COST, short=SHORT_COST):
    real, cantidad = np.asarray(real, float), np.asarray(cantidad, float)
    exceso, faltante = np.maximum(cantidad - real, 0), np.maximum(real - cantidad, 0)
    return {"MAE": mean_absolute_error(real, cantidad),
            "RMSE": mean_squared_error(real, cantidad) ** 0.5,
            "WAPE": np.abs(real - cantidad).sum() / real.sum() if real.sum() > 0 else np.nan,  # no definido si la demanda total es 0
            "unidades_excedentes": exceso.sum(), "unidades_faltantes": faltante.sum(),
            "costo": (exceso * over + faltante * short).sum()}

def resumir(filas, por, orden):
    """Resume los folds por modelo o por política. 'orden' va de la opción más simple a la más compleja."""
    t = (pd.DataFrame(filas).groupby(por)
           .agg(costo_prom=("costo", "mean"), costo_sd=("costo", "std"),
                MAE_prom=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_prom=("RMSE", "mean"),
                WAPE_prom=("WAPE", "mean"), excedentes=("unidades_excedentes", "sum"),
                faltantes=("unidades_faltantes", "sum"))
           .reindex(orden))
    t["complejidad"] = range(len(orden))
    return t

def seleccionar(tabla, tol=TOLERANCIA):
    """Regla única de selección (ver el texto de esta sección)."""
    minimo = tabla["costo_prom"].min()
    limite = minimo * (1 + tol)          # si minimo == 0, limite == 0: solo empatan costos 0
    candidatos = tabla[tabla["costo_prom"] <= limite + 1e-9]
    return candidatos.sort_values(["MAE_prom", "costo_sd", "complejidad"], kind="stable").index[0]
print("Funciones listas.")

Funciones listas.


# 9. Backtesting rolling-origin (solo con el entrenamiento)

Cada fold entrena con todo lo anterior a su origen y pronostica las **8 semanas** siguientes, igual que el test final. Se exige al menos 52 semanas de entrenamiento; los bloques de validación no se superponen y el último termina justo antes del test. La celda **comprueba** que ningún fold invada el test.

In [ ]:
BT_H, MIN_TRAIN = 8, 52
origenes = list(range(len(train) - BT_H, MIN_TRAIN - 1, -BT_H))[::-1]

def backtest(y, modelos, etiqueta):
    """Pronostica cada bloque de validación desde su origen. Devuelve métricas por fold y cantidades."""
    filas, cantidades = [], {}
    for k, origen in enumerate(origenes, start=1):
        tr, va = y.iloc[:origen], y.iloc[origen:origen + BT_H]
        assert tr.index[-1] < va.index[0] and va.index[-1] < semanas_test[0], "un fold invade el test"
        for m in modelos:
            cantidades[(k, m)] = pronosticar(m, tr, BT_H, f"{etiqueta} · fold {k}")
            filas.append({**evaluar(va, cantidades[(k, m)]), "fold": k, "modelo": m})
    return filas, cantidades

filas_bt, q_bt = backtest(train, MODELOS, "backtesting")
bt = pd.DataFrame(filas_bt)
folds = pd.DataFrame([{"fold": k, "entrenamiento": f"{train.index[0].date()} → {train.index[o - 1].date()}", "n_train": o,
                       "validacion": f"{train.index[o].date()} → {train.index[o + BT_H - 1].date()}"}
                      for k, o in enumerate(origenes, start=1)])
display(folds)
display(bt.round(2))
print("Comprobado: los", len(origenes), "folds terminan antes del test (", semanas_test[0].date(), ").")

,fold,entrenamiento,n_train,validacion
0,1,2009-12-07 → 2010-12-27,56,2011-01-03 → 2011-02-21
1,2,2009-12-07 → 2011-02-21,64,2011-02-28 → 2011-04-18
2,3,2009-12-07 → 2011-04-18,72,2011-04-25 → 2011-06-13
3,4,2009-12-07 → 2011-06-13,80,2011-06-20 → 2011-08-08
4,5,2009-12-07 → 2011-08-08,88,2011-08-15 → 2011-10-03


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo,fold,modelo
0,608.12,1044.62,1.00,0.0,4865.0,7297.5,1,Naive
1,528.88,758.45,0.87,1195.0,3036.0,5032.0,1,SeasonalNaive
2,1599.12,1838.12,2.63,11475.0,1318.0,6567.0,1,ExpSmoothing
3,1111.12,1460.84,0.83,113.0,8776.0,13209.2,2,Naive
4,789.88,1000.51,0.59,1317.0,5002.0,8029.8,2,SeasonalNaive
5,949.12,1147.68,0.71,4251.0,3342.0,6713.4,2,ExpSmoothing
6,723.88,824.92,0.77,4550.0,1241.0,3681.5,3,Naive
7,1248.88,1463.50,1.33,8557.0,1434.0,5573.8,3,SeasonalNaive
8,850.75,960.03,0.91,5384.0,1422.0,4286.6,3,ExpSmoothing
9,391.88,471.10,0.75,47.0,3088.0,4650.8,4,Naive


Comprobado: los 5 folds terminan antes del test ( 2011-10-10 ).


**Interpretación de resultados**

El output confirma la ejecución correcta del backtesting rolling-origin con 5 folds, donde la ventana de entrenamiento crece progresivamente de 56 a 88 semanas y la validación se mantiene constante en 8 semanas, garantizando que ningún bloque invada el periodo de prueba (finalizando estrictamente antes del 10 de octubre de 2011). Al analizar el desempeño económico, el modelo Naïve presenta una alta volatilidad, evidenciada por un costo extremo en el fold 2 (£13 209,2) debido a 8 776 unidades faltantes. Por el contrario, Exponential Smoothing demuestra una mayor estabilidad y adaptación a medida que aumenta el historial de entrenamiento, logrando los costos más bajos en los folds 4 y 5 (£2 534,0 y £1 519,5, respectivamente) al mitigar eficazmente las unidades faltantes. Esto valida la robustez de la metodología para evaluar el desempeño temporal de los modelos sin fuga de información.

In [ ]:
resumen_val = resumir(filas_bt, "modelo", MODELOS)
SELECCIONADO = seleccionar(resumen_val)
display(resumen_val.round(2))
print("MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test):", SELECCIONADO)
fig = px.box(bt, x="modelo", y="costo", points="all", title="Costo simulado por fold de validación (£)",
             labels={"modelo": "Modelo", "costo": "Costo por fold (£)"})
fig.show()

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,6239.22,4296.41,620.2,327.68,819.13,0.75,5469.0,19339.0,0
SeasonalNaive,4950.34,2065.83,666.5,370.02,830.61,0.83,13853.0,12807.0,1
ExpSmoothing,4324.10,2335.11,927.3,420.72,1063.63,1.29,30925.0,6167.0,2


MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test): ExpSmoothing


**Interpretación de resultados**

El gráfico de cajas y la tabla resumen evidencian una clara disociación entre el error estadístico y el costo económico de los modelos. ExpSmoothing fue seleccionado con el menor costo promedio de validación (£4,324.10), a pesar de registrar el mayor MAE (927.3) y RMSE (1,063.63); esto se explica por el costo asimétrico del error, ya que el modelo produce significativamente menos unidades faltantes (6,167) a cambio de un mayor volumen de excedentes (30,925), y dado que el faltante cuesta £1.50 frente a £0.40 del excedente, la reducción de faltantes compensa ampliamente el exceso. Por el contrario, Naïve presenta el mayor costo promedio (£6,239.22) y la mayor volatilidad entre folds (costo_sd £4,296.41), con un valor atípico extremo cercano a £13,200 en el fold 2, producto de su alta generación de faltantes (19,339). SeasonalNaive resulta ser el modelo más estable (costo_sd £2,065.83), pero no el más económico. En conjunto, la validación confirma que la regla de selección basada en el costo promedio captura adecuadamente el impacto económico real del error, priorizando la mitigación de faltantes sobre la minimización del error absoluto.

### Pregunta 4
¿Qué modelo se seleccionó en validación y con qué regla? ¿Coinciden el ranking por costo y el ranking por MAE o RMSE? ¿Por qué el ranking por WAPE no aporta un criterio adicional al MAE?

**Respuesta:**

En la validación se seleccionó el modelo Exponential Smoothing aplicando la regla única de selección: se eligió el candidato con el menor costo promedio de validación (dentro de una tolerancia del 1 % respecto al mínimo), con desempates por menor MAE, menor desviación del costo y menor complejidad. Los rankings no coinciden: mientras el ranking por costo ubica a Exponential Smoothing en primer lugar, los rankings por MAE y RMSE lo colocan en último lugar, siendo Naïve el mejor en ambas métricas estadísticas. Esta discrepancia se explica por el costo asimétrico del error; Exponential Smoothing genera más unidades excedentes (costo bajo de £0,40) y significativamente menos unidades faltantes (costo alto de £1,50) que Naïve, lo que reduce su impacto económico total a pesar de presentar un mayor error absoluto. Finalmente, el ranking por WAPE no aporta un criterio adicional al MAE porque, al compararse los mismos modelos sobre las mismas semanas, el WAPE es directamente proporcional al MAE (WAPE = MAE × semanas / demanda total), por lo que ambas métricas ordenan a los modelos de manera idéntica.


### Pregunta 5
¿Qué tan estable fue cada modelo entre folds? Use la desviación del costo y del MAE.

**Respuesta:**

Al evaluar la estabilidad entre los cinco folds de validación, el modelo SeasonalNaive demostró ser el más consistente en términos económicos, presentando la menor desviación estándar del costo (£2065.83), seguido por Exponential Smoothing (£2335.11). Por el contrario, el modelo Naïve exhibió la mayor volatilidad financiera (£4296.41), a pesar de registrar la menor desviación estándar del MAE (327.68 unidades); esto indica que, aunque la magnitud de su error absoluto es relativamente constante, la dirección de dicho error (exceso o faltante) fluctúa drásticamente entre periodos, impactando severamente el costo. Finalmente, Exponential Smoothing mostró la mayor variabilidad en la magnitud del error absoluto (MAE_sd = 420.72), lo que refleja una adaptación desigual a los distintos tramos de la serie temporal analizada.


# 10. Política de colchón calibrada sin información futura (antes del test)

El colchón suma unidades al pronóstico para protegerse de faltantes: **colchón = ⌈multiplicador × MAE previo⌉**. Se comparan los multiplicadores 0; 0,5; 1 y 1,5.

**Regla de calibración (sin errores futuros).** Para cada origen, el *MAE previo* se calcula con el modelo seleccionado en los **2 bloques de 8 semanas inmediatamente anteriores** a ese origen: cada bloque se pronostica entrenando solo con lo anterior a él. Así, el colchón del fold 1 usa solo errores anteriores al fold 1.
- Si no hay historia suficiente (algún bloque quedaría con menos de 16 semanas de entrenamiento), el colchón de ese origen es **0** y se informa. Con 52 semanas mínimas no ocurre en este laboratorio.
- La política (modelo, multiplicador y regla) se elige con la **regla única de selección** y queda **fijada antes del test**.
- Para el test, el MAE previo se calcula con la misma regla desde el final del entrenamiento (solo información previa al test).
- Después de evaluar, la **magnitud** del colchón se actualiza con la historia completa para proyectar, **sin cambiar la regla**.

In [ ]:
MULTIPLICADORES = [0, 0.5, 1.0, 1.5]
BLOQUES_CAL = 2

def mae_previo(modelo, y, etiqueta, bloques=BLOQUES_CAL):
    """MAE del modelo en los 'bloques' bloques de 8 semanas que terminan al final de y.
    Solo usa y (información hasta el origen). Devuelve None si la historia no alcanza."""
    errores = []
    for b in range(bloques, 0, -1):
        corte = len(y) - b * BT_H
        if corte < 2 * SEASON + 8:
            return None
        real = y.iloc[corte:corte + BT_H].values
        errores.append(np.abs(real - pronosticar(modelo, y.iloc[:corte], BT_H, f"{etiqueta} · bloque {b}")))
    return float(np.mean(np.concatenate(errores)))

def colchon(mult, mae):
    return 0 if (mae is None or mult == 0) else int(np.ceil(mult * mae))

def comparar_politicas(bloques=BLOQUES_CAL, etiqueta="calibración"):
    filas = []
    for k, origen in enumerate(origenes, start=1):
        tr, va = train.iloc[:origen], train.iloc[origen:origen + BT_H]
        mae_k = mae_previo(SELECCIONADO, tr, f"{etiqueta} · fold {k}", bloques)
        for mult in MULTIPLICADORES:
            c = colchon(mult, mae_k)
            filas.append({**evaluar(va, q_bt[(k, SELECCIONADO)] + c), "fold": k, "multiplicador": mult,
                          "mae_previo": mae_k, "colchon": c,
                          "ultima_semana_calibracion": tr.index[-1].date(), "inicio_validacion": va.index[0].date()})
    return pd.DataFrame(filas)

pol_folds = comparar_politicas()
assert (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all(), "colchón con información futura"
resumen_pol = resumir(pol_folds, "multiplicador", MULTIPLICADORES)
MULT = seleccionar(resumen_pol)

display(pol_folds.loc[pol_folds["multiplicador"] == 1.0,
                      ["fold", "ultima_semana_calibracion", "inicio_validacion", "mae_previo"]].round(1))
display(resumen_pol.round(2))
MAE_TEST = mae_previo(SELECCIONADO, train, "calibración test")       # solo entrenamiento
COLCHON_TEST = colchon(MULT, MAE_TEST)
POLITICA = f"{SELECCIONADO} + {MULT:g} × MAE previo"
print("POLÍTICA FIJADA ANTES DEL TEST:", POLITICA)
print(f"Colchón para el test: {COLCHON_TEST} unidades por semana (MAE previo = {MAE_TEST:.1f}, calculado solo con el entrenamiento)")

,fold,ultima_semana_calibracion,inicio_validacion,mae_previo
2,1,2010-12-27,2011-01-03,1400.7
6,2,2011-02-21,2011-02-28,1533.2
10,3,2011-04-18,2011-04-25,1274.1
14,4,2011-06-13,2011-06-20,899.9
18,5,2011-08-08,2011-08-15,821.3


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


POLÍTICA FIJADA ANTES DEL TEST: ExpSmoothing + 0 × MAE previo
Colchón para el test: 0 unidades por semana (MAE previo = 618.8, calculado solo con el entrenamiento)


**Interpretación de resultados**

La tabla de MAE previo confirma que la calibración del colchón se realizó correctamente sin fuga de información, utilizando exclusivamente los dos bloques de 8 semanas anteriores a cada origen de validación, con valores que fluctúan entre 821.3 y 1533.2 unidades según el historial disponible. El resumen por política revela que el multiplicador 0 (sin colchón) fue seleccionado por presentar el menor costo promedio de validación (£4,324.10), a pesar de generar el mayor número de unidades faltantes (6,167). Esto se explica porque los multiplicadores superiores, aunque reducen drásticamente los faltantes (hasta llegar a 0 con el multiplicador 1.5), generan un volumen excesivo de excedentes (95,926 unidades con k=1.5) cuyo costo acumulado (£0.40 c/u) supera el ahorro obtenido al evitar faltantes (£1.50 c/u). La política fijada para el test fue ExpSmoothing + 0 × MAE previo, con un colchón de 0 unidades semanales calculado sobre un MAE previo de 618.8 del entrenamiento, lo que indica que, para este SKU y bajo los supuestos de costos asimétricos del laboratorio, añadir un colchón de seguridad resultaba económicamente ineficiente en la validación.

### Pregunta 6
¿Qué política de colchón quedó fijada antes del test y por qué? Explique por qué el colchón de cada fold se calcula solo con errores anteriores a su origen.

**Respuesta:**

La política fijada antes del test fue ExpSmoothing con un multiplicador de 0 × MAE previo (es decir, sin colchón). Esta selección se fundamenta en la regla única de decisión, ya que esta opción presentó el menor costo promedio de validación (£4324.10), cumpliendo con la tolerancia del 1 % y ganando los desempates por menor MAE y menor complejidad frente a las demás alternativas. Por otro lado, el colchón de cada fold se calcula exclusivamente con los errores de los bloques anteriores a su origen para evitar la fuga de información (data leakage). En un escenario real, la calibración del stock de seguridad solo puede basarse en el desempeño histórico conocido hasta el momento de tomar la decisión; utilizar errores futuros invalidaría la honestidad de la evaluación temporal, pues simularía un conocimiento que no estaría disponible en la práctica operativa.


# 11. Evaluación final en test (una sola vez)

Se reentrena cada modelo con las 96 semanas de entrenamiento y se pronostican las 8 semanas de test. Se evalúan los tres modelos sin colchón y la **política fijada**. Si otra opción resulta mejor aquí, se informa, pero **no** cambia la selección.

In [ ]:
cantidades_test = {m: pronosticar(m, train, TEST_H, "test") for m in MODELOS}
q_politica = cantidades_test[SELECCIONADO] + COLCHON_TEST
filas = [{**evaluar(test, q), "opcion": f"{m} (sin colchón)"} for m, q in cantidades_test.items()]
if MULT > 0:
    filas.append({**evaluar(test, q_politica), "opcion": f"Política fijada: {POLITICA}"})
res_test = pd.DataFrame(filas).set_index("opcion")
display(res_test.round(2))

fila_pol = f"Política fijada: {POLITICA}" if MULT > 0 else f"{SELECCIONADO} (sin colchón)"
costo_pol, costo_naive = res_test.loc[fila_pol, "costo"], res_test.loc["Naive (sin colchón)", "costo"]
diferencia = costo_naive - costo_pol        # > 0 ahorro; < 0 sobrecosto
print("Política evaluada:", POLITICA)
if SELECCIONADO == "Naive" and MULT == 0:
    print("La política seleccionada es el propio Naïve: no hay diferencia que calcular frente a él.")
elif costo_naive == 0:
    print(f"Diferencia frente a Naïve: £{diferencia:.2f}. No se calcula porcentaje porque el costo de Naïve es cero.")
elif diferencia > 0:
    print(f"Ahorro frente a Naïve: £{diferencia:.2f} ({diferencia / costo_naive * 100:.1f} %).")
else:
    print(f"Sobrecosto frente a Naïve: £{-diferencia:.2f} ({-diferencia / costo_naive * 100:.1f} %).")
mejor_retro = res_test["costo"].idxmin()
if mejor_retro != fila_pol:
    print(f"Retrospectivamente, «{mejor_retro}» habría costado menos en test; la selección no cambia.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Ventas reales", line=dict(width=3)))
for m, q in cantidades_test.items():
    fig.add_trace(go.Scatter(x=test.index, y=q, mode="lines+markers", name=m))
fig.update_layout(title="Test final: ventas reales y cantidades operativas", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
Naive (sin colchón),678.75,1545.12,0.48,961.0,4469.0,7087.9
SeasonalNaive (sin colchón),773.75,1631.81,0.54,641.0,5549.0,8579.9
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6


Política evaluada: ExpSmoothing + 0 × MAE previo
Sobrecosto frente a Naïve: £1869.70 (26.4 %).
Retrospectivamente, «Naive (sin colchón)» habría costado menos en test; la selección no cambia.


**Interpretación de resultados**

El gráfico y la tabla del test final evidencian que la política seleccionada (ExpSmoothing + 0 × MAE previo) no logró superar al baseline Naïve, generando un sobrecosto de £1869.70 (26.4 %) frente a las £7087.90 de Naïve. La causa principal se observa en el gráfico: la semana del 23 de octubre de 2011 registró un pico extraordinario de aproximadamente 5300 unidades, que ningún modelo univariante pudo anticipar, ya que los tres pronósticos se mantuvieron entre 500 y 1000 unidades. Esta subestimación masiva se tradujo en 5884 unidades faltantes para ExpSmoothing, el mayor volumen entre los tres modelos, lo que explica su RMSE más alto (1700.86) y su costo total más elevado (£8957.60). Aunque ExpSmoothing produjo menos unidades excedentes (329) que Naïve (961), el costo asimétrico del error (£1.50 por faltante vs £0.40 por excedente) hizo que la reducción de excedentes no compensara el incremento de faltantes. Retrospectivamente, Naïve habría sido la opción más económica en el test, pero la selección fijada en validación no se modifica, confirmando que el desempeño en validación no siempre se replica en el test, especialmente frente a eventos atípicos no modelados.

### Pregunta 7
¿La política seleccionada cumplió en test? Compare su costo con los baselines e indique el ahorro o sobrecosto frente a Naïve. ¿El modelo con menor RMSE fue también el de menor costo?

**Respuesta:**

La política seleccionada no cumplió su objetivo en el test, ya que no logró superar al baseline. El costo de la política fijada (ExpSmoothing sin colchón) fue de £8957.60, lo que representó un sobrecosto de £1869.70 (26.4 %) frente a Naïve, cuyo costo total fue de £7087.90. Respecto a las métricas de error, el modelo con menor RMSE fue efectivamente Naïve (1545.12), el cual también registró el menor costo; esto confirma que, en esta evaluación final, la mayor complejidad de Exponential Smoothing no se tradujo en un mejor desempeño ni estadístico ni económico, validando la decisión de no adoptarlo como sustituto del método simple.


### Diagnóstico del pico del test (retrospectivo)

Se identifica la semana de mayor venta del test y el pedido más grande de esa semana. Un pedido extraordinario **aislado** es un **pico puntual**; un **quiebre estructural** exige un cambio persistente de nivel o patrón, que 8 semanas no permiten confirmar. Los modelos usados son **univariantes**: solo ven las ventas pasadas del SKU y no disponían de información específica sobre ese pedido.

La evaluación principal **conserva** esa semana. El cálculo sin ella es un **análisis retrospectivo de sensibilidad**, no una evaluación alternativa.

In [ ]:
semana_pico = test.idxmax()
lineas_pico = uk[(uk["StockCode"] == SKU) & (uk["week"] == semana_pico)]
pedidos = lineas_pico.groupby(["InvoiceNo", "CustomerID"], dropna=False)["Quantity"].sum().sort_values(ascending=False)
print(f"Semana de mayor venta en test: {semana_pico.date()} · {test.max():.0f} unidades")
factura, cliente = pedidos.index[0]
cliente = "sin dato" if pd.isna(cliente) else str(cliente).removesuffix(".0")
print(f"Pedido más grande: factura {factura} · cliente {cliente} · {pedidos.iloc[0]:.0f} unidades "
      f"({pedidos.iloc[0] / test.max() * 100:.1f} % de la semana)")
print("Mediana semanal del entrenamiento:", train.median(), "unidades")

sin_pico = test.index != semana_pico
sens = pd.DataFrame({m: evaluar(test[sin_pico], cantidades_test[m][sin_pico]) for m in MODELOS}).T
if MULT > 0:
    sens.loc["Política fijada"] = evaluar(test[sin_pico], q_politica[sin_pico])
print("\nANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.")
display(sens[["MAE", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

Semana de mayor venta en test: 2011-10-24 · 5328 unidades
Pedido más grande: factura 573008 · cliente 12901 · 4800 unidades (90.1 % de la semana)
Mediana semanal del entrenamiento: 584.5 unidades

ANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.


,MAE,unidades_excedentes,unidades_faltantes,costo
Naive,155.86,961.0,130.0,579.4
SeasonalNaive,232.86,641.0,989.0,1739.9
ExpSmoothing,205.86,329.0,1112.0,1799.6


**Interpretación de resultados**

El análisis del pico de test confirma que la semana del 24 de octubre de 2011, con 5328 unidades, constituye un pico puntual y no un quiebre estructural, dado que un único pedido (factura 573008) concentró el 90,1 % de ese volumen, alejándose drásticamente de la mediana semanal del entrenamiento (584,5 unidades). El análisis retrospectivo de sensibilidad, al excluir esta semana atípica, muestra una reducción drástica en los costos de todos los modelos (por ejemplo, Naïve baja a £579,40), lo que evidencia que el sobrecosto de la evaluación principal fue impulsado casi exclusivamente por este evento extraordinario que los modelos univariantes no podían anticipar. No obstante, esta sensibilidad no invalida la evaluación principal, ya que el test debe reflejar el desempeño real frente a la operativa; además, confirma que incluso en un escenario "normal" de 7 semanas, Naïve sigue siendo la opción más económica (£579,40) frente a ExpSmoothing (£1799,60), reforzando la decisión de no adoptar el modelo complejo como sustituto del baseline.

### Pregunta 8
¿Qué error fue más grave en el test: sobreestimar o subestimar? Use las unidades y los costos. ¿El pico del test indica un quiebre estructural o un pico puntual? ¿Qué muestra, y qué no, el análisis de sensibilidad sin esa semana?

**Respuesta:**

En el periodo de test, el error de subestimar la demanda fue significativamente más grave, generando 5884 unidades faltantes frente a solo 329 excedentes; esto se tradujo en un costo de aproximadamente £8826 por faltantes (5884 × £1,50), que representa la inmensa mayoría del costo total de £8957,60 del modelo seleccionado. El pico observado en la semana del 24 de octubre de 2011 (5328 unidades) se clasifica como un pico puntual y no como un quiebre estructural, dado que el 90,1 % de ese volumen provino de un único pedido extraordinario (factura 573008), muy alejado de la mediana semanal del entrenamiento (584,5 unidades), sin evidencia de un cambio persistente en el nivel de la serie. Finalmente, el análisis de sensibilidad sin esa semana muestra cómo mejorarían drásticamente las métricas de error y costo en un escenario "normal" (por ejemplo, reduciendo el costo de Naïve a £579,40), pero no constituye una evaluación alternativa válida para revertir la decisión, ya que los modelos univariantes no disponían de información previa sobre dicho evento y el test debe reflejar el desempeño frente a la realidad operativa tal como ocurrió.


# 12. Reentrenamiento final y proyección académica

Después de cerrar la evaluación se reentrena el modelo seleccionado con **toda** la historia (104 semanas) y se actualiza la **magnitud** del colchón con la misma regla (los 2 bloques más recientes, que ahora incluyen el test). Se muestra también la proyección de Naïve como referencia.

La tabla es una **proyección académica**, no una orden de compra:
- La **cantidad objetivo semanal** es un objetivo para cada semana. El laboratorio **no simula el arrastre de inventario** entre semanas: si una semana sobra mercadería, esta reduce la necesidad de la siguiente. Por eso la suma de los objetivos semanales no equivale a la cantidad a comprar.
- Una compra simplificada para una semana sería `max(objetivo − inventario utilizable − pedidos que llegarán a tiempo, 0)`: nunca negativa, y solo cuenta el inventario apto para la venta y los pedidos que llegarán antes de esa semana. El laboratorio no dispone de esos datos.

In [ ]:
FINAL_H = 8
pron_final = pronosticar(SELECCIONADO, serie, FINAL_H, "final")
pron_naive = pronosticar("Naive", serie, FINAL_H, "final")
MAE_FINAL = mae_previo(SELECCIONADO, serie, "calibración final")   # misma regla, historia completa
COLCHON_FINAL = colchon(MULT, MAE_FINAL)
semanas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=FINAL_H, freq="W-MON")
plan = pd.DataFrame({"Semana": semanas_futuras.date,
                     f"Pronóstico {SELECCIONADO}": pron_final.astype(int),
                     "Colchón (política fijada)": COLCHON_FINAL,
                     "Cantidad objetivo semanal": (pron_final + COLCHON_FINAL).astype(int),
                     "Referencia Naïve": pron_naive.astype(int)})
display(plan)
print(f"Política aplicada: {POLITICA} · MAE previo actualizado = {MAE_FINAL:.1f} · colchón = {COLCHON_FINAL} unidades por semana")
print("Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.")

,Semana,Pronóstico ExpSmoothing,Colchón (política fijada),Cantidad objetivo semanal,Referencia Naïve
0,2011-12-05,1075,0,1075,689
1,2011-12-12,899,0,899,689
2,2011-12-19,1617,0,1617,689
3,2011-12-26,998,0,998,689
4,2012-01-02,1103,0,1103,689
5,2012-01-09,927,0,927,689
6,2012-01-16,1645,0,1645,689
7,2012-01-23,1027,0,1027,689


Política aplicada: ExpSmoothing + 0 × MAE previo · MAE previo actualizado = 611.1 · colchón = 0 unidades por semana
Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.


**Interpretación de resultados**

La tabla presenta la proyección académica a 8 semanas tras reentrenar el modelo Exponential Smoothing con la historia completa de 104 semanas. La política de cantidad se mantiene estrictamente en un colchón de 0 unidades, dado que el multiplicador seleccionado fue 0, a pesar de que el MAE previo actualizado asciende a 611.1 unidades. Los objetivos semanales fluctúan entre 899 y 1645 unidades, reflejando la capacidad del modelo para capturar los patrones estacionales (con picos notables a mediados de diciembre y enero), a diferencia de la referencia Naïve, que se mantiene constante en 689 unidades al repetir la última observación. Es fundamental destacar que esta proyección no constituye una orden de compra real, ya que el laboratorio no simula el arrastre de inventario entre semanas ni descuenta el stock utilizable o los pedidos en tránsito, variables indispensables para calcular la necesidad neta de abastecimiento en un entorno operativo.


# 13. Decisión de adopción y resumen automático

La decisión se toma con una regla fijada de antemano:
- **Adoptar** (como piloto controlado): la política seleccionada costó menos que Naïve en el test **y** en la mayoría de los folds de validación.
- **Mantener en evaluación:** superó a Naïve en el test, pero no de forma consistente en validación.
- **No adoptar como sustituto de Naïve:** en el test costó igual o más que Naïve. Naïve sigue como referencia; el modelo puede seguir evaluándose sin decidir compras con él.

Un **piloto** sirve para comprobar, durante varias semanas nuevas y con la misma regla, si la política reduce el costo frente a Naïve; no justifica por sí mismo una cantidad de compra.

In [ ]:
naive_bt = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]
pol_bt = pol_folds[pol_folds["multiplicador"] == MULT].set_index("fold")["costo"]
folds_ganados = int((pol_bt < naive_bt).sum())
if costo_pol < costo_naive and folds_ganados > len(origenes) / 2:
    DECISION = "Adoptar como piloto controlado frente a Naïve"
elif costo_pol < costo_naive:
    DECISION = "Mantener en evaluación"
else:
    DECISION = "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

resumen = {
    "Producto analizado": f"{SKU} · {desc}",
    "Seleccionado antes del test": f"{SELECCIONADO}; política {POLITICA} (regla única de selección)",
    "Costo promedio de validación de la política": f"£{resumen_pol.loc[MULT, 'costo_prom']:.2f} ± £{resumen_pol.loc[MULT, 'costo_sd']:.2f} ({len(origenes)} folds)",
    "Folds de validación en que superó a Naïve": f"{folds_ganados} de {len(origenes)}",
    "Resultado en test (8 semanas)": f"£{costo_pol:.2f} frente a £{costo_naive:.2f} de Naïve",
    "Frente a Naïve": ("ahorro" if diferencia > 0 else "sobrecosto") + f" de £{abs(diferencia):.2f}" + (f" ({abs(diferencia) / costo_naive * 100:.1f} %)" if costo_naive > 0 else ""),
    "Unidades excedentes / faltantes en test": f"{res_test.loc[fila_pol, 'unidades_excedentes']:.0f} / {res_test.loc[fila_pol, 'unidades_faltantes']:.0f}",
    "Decisión": DECISION,
    "Proyección académica (suma de objetivos semanales)": f"{int(plan['Cantidad objetivo semanal'].sum())} unidades; no es una orden de compra",
}
for k, v in resumen.items():
    print(f"{k}: {v}")

Producto analizado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS
Seleccionado antes del test: ExpSmoothing; política ExpSmoothing + 0 × MAE previo (regla única de selección)
Costo promedio de validación de la política: £4324.10 ± £2335.11 (5 folds)
Folds de validación en que superó a Naïve: 4 de 5
Resultado en test (8 semanas): £8957.60 frente a £7087.90 de Naïve
Frente a Naïve: sobrecosto de £1869.70 (26.4 %)
Unidades excedentes / faltantes en test: 329 / 5884
Decisión: No adoptar como sustituto de Naïve; Naïve se mantiene como referencia
Proyección académica (suma de objetivos semanales): 9291 unidades; no es una orden de compra


**Interpretación de resultados**

El resumen ejecutivo confirma que, a pesar de que la política de Exponential Smoothing con un colchón de 0 × MAE previo fue seleccionada en validación por su menor costo promedio (£4324.10) y su consistencia al superar a Naïve en 4 de 5 folds, no logró generalizar este desempeño en el test final. Durante las 8 semanas de prueba, la política incurrió en un sobrecosto del 26.4 % (£1869.70) frente al baseline Naïve (£7087.90), impulsado principalmente por una subestimación masiva de la demanda (5884 unidades faltantes frente a solo 329 excedentes), probablemente debido a un pico atípico no anticipado por los modelos univariantes. En consecuencia, la decisión fundamentada es no adoptar el modelo complejo como sustituto, manteniendo a Naïve como la referencia operativa. Asimismo, se reitera que la proyección académica de 9291 unidades para las próximas semanas es un objetivo teórico semanal y no constituye una orden de compra real, ya que no descuenta el inventario utilizable ni los pedidos en tránsito.

### Pregunta 9 — Cierre ejecutivo
Cierre ejecutivo (máximo 12 líneas). Distinga: modelo y política seleccionados antes del test; resultado de su evaluación frente a Naïve; decisión (adoptar, mantener en evaluación o no adoptar) y su fundamento; proyección académica y por qué no es una orden de compra; una condición que cambiaría la decisión; y las limitaciones (supuestos de costos, ceros, disponibilidad y un solo test de 8 semanas).

**Respuesta:**

El modelo seleccionado antes del test fue ExpSmoothing, con la política ExpSmoothing + 0 × MAE previo (costo promedio de validación £4324.10 ± £2335.11). En el test de 8 semanas, esta política costó £8957.60 frente a £7087.90 de Naïve, generando un sobrecosto de £1869.70 (26.4 %), por lo que la decisión es no adoptar como sustituto de Naïve, manteniendo Naïve como referencia. La proyección académica de 9291 unidades no constituye una orden de compra porque el laboratorio no simula el arrastre de inventario entre semanas ni descuenta el inventario utilizable o los pedidos que llegarían a tiempo. La decisión cambiaría si la política hubiera costado menos que Naïve en el test y en la mayoría de los folds de validación. Las limitaciones incluyen los costos asimétricos supuestos (£0.40 por exceso y £1.50 por faltante, no provenientes de UCI), el tratamiento de ceros como supuesto sobre ventas observadas sin distinguir demanda nula, falta de inventario o ausencia de registros, la no verificación de disponibilidad real del producto, y la evaluación basada en un único test de 8 semanas, donde un pico puntual (como el pedido de 4800 unidades) puede distorsionar el resultado.


# 14. Retos individuales de aplicación

La parte guiada ya desarrolló los procedimientos. En esta sección los aplicará a tres casos empresariales. Cada reto tiene cuatro apartados: **A** plantear o preparar el análisis, **B** implementar y obtener resultados, **C** comparar e interpretar la evidencia y **D** formular una decisión justificada con sus limitaciones. Reutilice las funciones de la parte guiada (`evaluar`, `resumir`, `seleccionar`, `backtest`, `pronosticar`, `mae_previo`, `colchon` y `comparar_politicas`) y los objetos ya calculados (`train`, `test`, `origenes`, `q_bt`, `cantidades_test`, `elegibles`). No modifique datos, semanas, pronósticos ni el test. Las respuestas deben ser breves, pero suficientes para justificar la decisión con cifras.

> **Entrega.** «Resuelva los tres retos y sus apartados A, B, C y D en su notebook personal. Guarde la versión ejecutada, con resultados e interpretaciones, en la carpeta semana10/ de su repositorio personal de GitHub y en su propio Google Colab. En la actividad de la semana, en el espacio habilitado, presente ambos enlaces: GitHub y Colab. Verifique que permitan la revisión del trabajo.»

## Reto 1 — Una campaña cambia el costo de los faltantes

**Caso.** La empresa prepara una campaña comercial. Durante la campaña, quedarse sin producto significa perder ventas y clientes que llegaron por la promoción, mientras que guardar unidades sobrantes cuesta menos. El responsable de abastecimiento necesita saber si la recomendación del modelo cambia cuando el faltante se vuelve más costoso.

**Problema.** ¿Se elegiría el mismo método de pronóstico bajo distintos costos del error y, con la evidencia del test, cuál debería usar abastecimiento como referencia?

**Datos, supuestos y recursos**
- Producto y semanas del laboratorio; pronósticos de validación ya calculados (`q_bt`) y cantidades de test (`cantidades_test`).
- Escenarios de costo (supuestos académicos, no provienen de UCI): A faltante £1,00 y exceso £0,50; B faltante £1,50 y exceso £0,40 (base); C faltante £3,00 y exceso £0,25.
- Regla común de selección (`resumir` y `seleccionar`). No se modifican ventas ni pronósticos para favorecer un resultado.

**Apartados**
- **A.** Plantear los escenarios y la decisión
- **B.** Calcular los costos de validación y seleccionar
- **C.** Evaluar en test y comparar frente a Naïve
- **D.** Recomendar a abastecimiento

**Evidencias que deben quedar en el notebook**
- Tabla de escenarios con la razón faltante / exceso.
- Tabla de costos promedio de validación por escenario y modelo, con el seleccionado.
- Tabla de test con excedentes, faltantes, costo y diferencia frente a Naïve, y su gráfico.
- Interpretaciones A–D.

### Reto 1 · Apartado A — Plantear los escenarios y la decisión
Construya una tabla con los tres escenarios y la razón faltante / exceso. Explique qué representa cada escenario para el negocio y formule la decisión empresarial que se tomará con el análisis.

In [ ]:
import pandas as pd

ESCENARIOS = {"A": (1.00, 0.50), "B": (1.50, 0.40), "C": (3.00, 0.25)}   # (faltante, exceso) en £ por unidad

# Construcción de la tabla de escenarios con la razón faltante / exceso
data_escenarios = []
for esc, (short, over) in ESCENARIOS.items():
    data_escenarios.append({
        "Escenario": esc,
        "faltante": short,
        "exceso": over,
        "faltante / exceso": round(short / over, 2)
    })

r1_escenarios = pd.DataFrame(data_escenarios).set_index("Escenario")
display(r1_escenarios)

,faltante,exceso,faltante / exceso
Escenario,,,
A,1.0,0.50,2.00
B,1.5,0.40,3.75
C,3.0,0.25,12.00


**Interpretación 1A:**

La razón "faltante / exceso" cuantifica la penalización relativa de subestimar la demanda frente a sobreestimarla, actuando como un indicador del nivel de servicio implícito que la empresa requiere. El Escenario A (razón 2,00) representa un entorno promocional moderado, donde el costo de una venta perdida es el doble que el de mantener un excedente. El Escenario B (razón 3,75) corresponde a la línea base del laboratorio, asumiendo que un faltante es casi cuatro veces más costoso que un exceso. El Escenario C (razón 12,00) simula una campaña de alta criticidad, donde perder un cliente por desabastecimiento es extremadamente costoso, mientras que el costo de mantener inventario sobrante es mínimo. La decisión empresarial que se tomará con este análisis es determinar si el incremento en la asimetría de costos modifica el modelo de pronóstico óptimo seleccionado en validación, o si el método base (Naïve) se mantiene como la referencia más robusta económicamente ante cualquier estructura de costos.


### Reto 1 · Apartado B — Calcular los costos de validación y seleccionar
Para cada escenario, recalcule el costo de cada fold con los pronósticos ya disponibles (`q_bt`), resuma por modelo y aplique la regla común de selección. Presente el costo promedio de validación de los tres modelos y el seleccionado en cada escenario.

In [ ]:
r1_val, r1_eleccion = {}, {}

for esc, (short, over) in ESCENARIOS.items():
    filas = []
    # Recalcular el costo de cada fold con los pronósticos ya disponibles (q_bt)
    for k, origen in enumerate(origenes, start=1):
        real = train.iloc[origen : origen + BT_H]
        for m in MODELOS:
            cantidad = q_bt[(k, m)]
            res = evaluar(real, cantidad, over=over, short=short)
            res["fold"] = k
            res["modelo"] = m
            filas.append(res)

    # Resumir por modelo y aplicar la regla común de selección
    r1_val[esc] = resumir(filas, "modelo", MODELOS)
    r1_eleccion[esc] = seleccionar(r1_val[esc])

# Construir la tabla resumen con el costo promedio por escenario y modelo + columna "seleccionado"
data_r1b = {}
for esc in ESCENARIOS.keys():
    data_r1b[esc] = {
        "Naive": r1_val[esc].loc["Naive", "costo_prom"],
        "SeasonalNaive": r1_val[esc].loc["SeasonalNaive", "costo_prom"],
        "ExpSmoothing": r1_val[esc].loc["ExpSmoothing", "costo_prom"],
        "seleccionado": r1_eleccion[esc]
    }

tabla_r1b = pd.DataFrame(data_r1b).T
tabla_r1b.index.name = "Escenario"
display(tabla_r1b.round(2))

,Naive,SeasonalNaive,ExpSmoothing,seleccionado
Escenario,,,,
A,4414.7,3946.7,4325.9,SeasonalNaive
B,6239.22,4950.34,4324.1,ExpSmoothing
C,11876.85,8376.85,5246.45,ExpSmoothing


**Interpretación 1B:**

La tabla de costos de validación evidencia que el modelo óptimo no es estático, sino que depende directamente de la estructura de costos del negocio. En el Escenario A, donde la penalización por faltantes es moderada (razón 2,00), SeasonalNaive resulta ser la opción más económica (£3946,70), logrando un mejor equilibrio general entre excedentes y faltantes. Sin embargo, en los Escenarios B y C, a medida que el costo del faltante se vuelve severo (razones 3,75 y 12,00), ExpSmoothing se consolida como el modelo seleccionado (£4324,10 y £5246,45, respectivamente). Esto se debe a que, aunque ExpSmoothing suele presentar un mayor error estadístico absoluto (MAE/RMSE), genera significativamente menos unidades faltantes; bajo una asimetría de costos elevada, evitar el quiebre de stock compensa ampliamente el costo de los excedentes. Este comportamiento valida que la decisión de abastecimiento debe guiarse por el impacto económico del error y no exclusivamente por la precisión estadística del pronóstico.


### Reto 1 · Apartado C — Evaluar en test y comparar frente a Naïve
Evalúe en test el modelo seleccionado en cada escenario (la selección ya quedó definida con validación) y Naïve con los mismos costos. Construya una tabla con excedentes, faltantes, costo y diferencia frente a Naïve, y un gráfico de barras. Interprete qué escenario y qué tipo de error explican las diferencias.

In [ ]:
import plotly.express as px

resultados_r1c = []

for esc, (short, over) in ESCENARIOS.items():
    modelo_sel = r1_eleccion[esc]

    # Evaluar el modelo seleccionado con los costos del escenario
    res_sel = evaluar(test, cantidades_test[modelo_sel], over=over, short=short)

    # Evaluar Naïve con los mismos costos del escenario
    res_naive = evaluar(test, cantidades_test["Naive"], over=over, short=short)

    resultados_r1c.append({
        "Escenario": esc,
        "seleccionado": modelo_sel,
        "excedentes": res_sel["unidades_excedentes"],
        "faltantes": res_sel["unidades_faltantes"],
        "costo_test": res_sel["costo"],
        "costo_test_Naive": res_naive["costo"],
        "diferencia_vs_Naive": res_naive["costo"] - res_sel["costo"]  # Positivo = ahorro
    })

tabla_r1c = pd.DataFrame(resultados_r1c).set_index("Escenario")
display(tabla_r1c.round(2))

# Gráfico de barras para visualizar la diferencia de costos
fig = px.bar(
    tabla_r1c.reset_index(),
    x="Escenario",
    y="diferencia_vs_Naive",
    color="diferencia_vs_Naive",
    color_continuous_scale=["red", "green"],
    title="Ahorro (positivo) o Sobrecosto (negativo) del modelo seleccionado frente a Naïve en el test",
    labels={"diferencia_vs_Naive": "Diferencia de costo (£)", "Escenario": "Escenario de costos"},
    text_auto='.2f'
)
fig.update_traces(textposition='outside')
fig.show()

,seleccionado,excedentes,faltantes,costo_test,costo_test_Naive,diferencia_vs_Naive
Escenario,,,,,,
A,SeasonalNaive,641.0,5549.0,5869.50,4949.50,-920.0
B,ExpSmoothing,329.0,5884.0,8957.60,7087.90,-1869.7
C,ExpSmoothing,329.0,5884.0,17734.25,13647.25,-4087.0


**Interpretación 1C:**

La evaluación en test revela que, bajo los tres escenarios de costos, el modelo seleccionado en validación generó sobrecosto frente a Naïve, evidenciando que ni la asimetría de costos ni la complejidad del modelo lograron revertir el desempeño del baseline en este periodo. El tipo de error dominante en todos los escenarios fue la subestimación de la demanda, con entre 5549 y 5884 unidades faltantes frente a solo 329-641 excedentes, lo cual se explica por el pico puntual de la semana del 24 de octubre de 2011 (5328 unidades) que ningún modelo univariante pudo anticipar. La magnitud del sobrecosto crece con la asimetría de costos: en el Escenario A (razón 2,00) el sobrecosto es de £920 (18,6 %), en el B (razón 3,75) asciende a £1869,70 (26,4 %) y en el C (razón 12,00) alcanza £4087 (30 %), lo que demuestra que, cuando el costo del faltante se vuelve extremo, la penalización por subestimar se amplifica drásticamente y afecta con mayor severidad a ExpSmoothing, que fue el modelo seleccionado en los escenarios B y C. En consecuencia, las diferencias observadas no se explican por la elección del modelo, sino por la ocurrencia de un evento atípico no modelable que castiga de forma desproporcionada a cualquier política que no incorpore información exógena sobre promociones o pedidos extraordinarios.


### Reto 1 · Apartado D — Recomendar a abastecimiento
Redacte una recomendación para abastecimiento: ¿cambia la selección entre escenarios?, ¿qué método usaría como referencia y por qué?, ¿qué límites tiene la conclusión? Distinga el ahorro simulado de un beneficio real demostrado.

**Decisión 1D:**

La recomendación para abastecimiento es mantener a Naïve como el método de referencia principal, a pesar de que la selección del modelo óptimo varía según la asimetría de costos (SeasonalNaive en el Escenario A y Exponential Smoothing en los Escenarios B y C). Esta decisión se fundamenta en que, durante la evaluación final en el test, ninguno de los modelos complejos logró superar a Naïve debido a un pico de demanda atípico e impredecible, lo que generó sobrecostos en todos los escenarios al no poder anticipar el evento. Las conclusiones tienen como límite principal estar basadas en un único periodo de prueba de 8 semanas dominado por un pedido extraordinario (4800 unidades) y en costos unitarios que son supuestos académicos, no datos reales de la empresa. Por lo tanto, es crucial distinguir que los ahorros o sobrecostos calculados son simulaciones retrospectivas del desempeño del modelo bajo reglas fijas, y no constituyen un beneficio financiero real demostrado, el cual solo podría validarse mediante un piloto controlado en operaciones futuras con los costos logísticos y de oportunidad reales del negocio.


## Reto 2 — ¿Conviene añadir un colchón?

**Caso.** El área comercial propone planificar unidades adicionales cada semana para reducir faltantes. Almacén advierte que el excedente también cuesta: espacio, capital inmovilizado y posibles liquidaciones. Se debe decidir con evidencia si conviene añadir un colchón al pronóstico.

**Problema.** ¿Qué política de cantidad, sin colchón o con colchón de 0,5; 1 o 1,5 veces el MAE previo, debería usarse, y qué costo produjo en test frente a no usar colchón?

**Datos, supuestos y recursos**
- Modelo seleccionado en la parte guiada (`SELECCIONADO`) y sus pronósticos de validación y test.
- Colchón = ⌈multiplicador × MAE previo⌉; el MAE previo usa solo los 2 bloques de 8 semanas anteriores a cada origen (`mae_previo`, `colchon`, `comparar_politicas`).
- Costos base: £1,50 por faltante y £0,40 por exceso (supuestos académicos). Regla común de selección.

**Apartados**
- **A.** Definir pronóstico, cantidad planificada y políticas
- **B.** Comparar las políticas en validación
- **C.** Seleccionar el multiplicador y evaluarlo en test
- **D.** Recomendar sobre el colchón

**Evidencias que deben quedar en el notebook**
- Tabla del MAE previo y del colchón de cada política por fold.
- Costos por fold y resumen por política con excedentes, faltantes y folds en que el colchón mejoró, más el gráfico de excedentes y faltantes.
- Tabla de test de la política seleccionada frente a la opción sin colchón.
- Interpretaciones A–D.

### Reto 2 · Apartado A — Definir pronóstico, cantidad planificada y políticas
Explique la diferencia entre pronóstico y cantidad planificada. Calcule, para cada fold, el MAE previo (solo con semanas anteriores a su origen) y el colchón que tendría cada política (0; 0,5; 1 y 1,5 × MAE previo).

In [ ]:
# MULTIPLICADORES = [0, 0.5, 1.0, 1.5] ya está definido en la parte guiada

filas_prep = []
for k, o in enumerate(origenes, start=1):
    # Calculamos el MAE previo usando solo la historia anterior al origen del fold
    mae_k = mae_previo(SELECCIONADO, train.iloc[:o], f"reto2 · fold {k}")

    # Calculamos el colchón para cada multiplicador
    for m in MULTIPLICADORES:
        filas_prep.append({
            "fold": k,
            "origen_semana": o,
            "multiplicador": m,
            "mae_previo": mae_k,
            "colchon": colchon(m, mae_k)
        })

r2_prep = pd.DataFrame(filas_prep)
display(r2_prep.round(1))

,fold,origen_semana,multiplicador,mae_previo,colchon
0,1,56,0.0,1400.7,0
1,1,56,0.5,1400.7,701
2,1,56,1.0,1400.7,1401
3,1,56,1.5,1400.7,2102
4,2,64,0.0,1533.2,0
5,2,64,0.5,1533.2,767
6,2,64,1.0,1533.2,1534
7,2,64,1.5,1533.2,2300
8,3,72,0.0,1274.1,0
9,3,72,0.5,1274.1,638


**Interpretación 2A:**

La tabla confirma la correcta calibración dinámica del colchón de seguridad, calculando el MAE previo exclusivamente con la historia disponible antes de cada origen de validación (descendiendo de 1400,7 en el fold 1 a 821,3 en el fold 5, a medida que el modelo se estabiliza con más datos). Al aplicar los multiplicadores (0; 0,5; 1,0 y 1,5), se evidencia una relación proporcional directa que transforma una misma estimación estadística en diferentes objetivos operativos, escalando el colchón desde 0 hasta 2102 unidades. Esto valida el procedimiento para establecer el trade-off fundamental entre el riesgo de quiebre de stock (que disminuye al aumentar el multiplicador) y el costo financiero de mantener inventario excedente (que aumenta proporcionalmente), garantizando que la política de cantidad se adapte al desempeño reciente del modelo sin incurrir en fuga de información.


### Reto 2 · Apartado B — Comparar las políticas en validación
Evalúe las cuatro políticas en los folds de validación con calibración temporal (`comparar_politicas`). Presente el costo por fold, el resumen por política (costo promedio, desviación, excedentes y faltantes), cuántos folds mejoró cada colchón frente a no usarlo y un gráfico de excedentes y faltantes.

In [ ]:
import plotly.graph_objects as go

# Evaluar las cuatro políticas en los folds de validación
r2_folds = comparar_politicas(etiqueta="reto2 · validación")

# Resumir por política
r2_resumen = resumir(r2_folds, "multiplicador", MULTIPLICADORES)

# Tabla de costo por fold (pivot)
costo_por_fold = r2_folds.pivot(index="fold", columns="multiplicador", values="costo")

# Conteo de folds en que cada multiplicador costó menos que 0×
costo_0x = r2_folds[r2_folds["multiplicador"] == 0.0].set_index("fold")["costo"]
mejoras_vs_0x = {
    mult: int((r2_folds[r2_folds["multiplicador"] == mult].set_index("fold")["costo"] < costo_0x).sum())
    for mult in MULTIPLICADORES if mult > 0
}

print("=== Resumen por política de colchón ===")
display(r2_resumen.round(2))

print("\n=== Costo por fold (£) ===")
display(costo_por_fold.round(2))

print("\nFolds en que el colchón mejoró el costo frente a 0×:", mejoras_vs_0x)

# Gráfico de excedentes y faltantes por multiplicador
fig = go.Figure()
fig.add_trace(go.Bar(
    x=r2_resumen.index.astype(str),
    y=r2_resumen["excedentes"],
    name="Unidades excedentes",
    marker_color="lightblue"
))
fig.add_trace(go.Bar(
    x=r2_resumen.index.astype(str),
    y=r2_resumen["faltantes"],
    name="Unidades faltantes",
    marker_color="lightcoral"
))
fig.update_layout(
    title="Excedentes y faltantes por política de colchón (validación)",
    xaxis_title="Multiplicador",
    yaxis_title="Unidades",
    barmode="group"
)
fig.show()

=== Resumen por política de colchón ===


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3



=== Costo por fold (£) ===


multiplicador,0.0,0.5,1.0,1.5
fold,,,,
1,6567.0,7478.3,8546.0,10789.2
2,6713.4,5345.0,5555.5,7723.6
3,4286.6,4563.1,5664.8,7703.2
4,2534.0,3974.0,5414.0,6854.0
5,1519.5,2673.2,3988.4,5300.4



Folds en que el colchón mejoró el costo frente a 0×: {0.5: 1, 1.0: 1, 1.5: 0}


**Interpretación 2B:**

El resumen de validación evidencia un trade-off directo y predecible: a medida que el multiplicador aumenta de 0 a 1,5, las unidades faltantes disminuyen drásticamente (de 6167 a 0), pero las unidades excedentes se disparan (de 30 925 a 95 926), lo que eleva el costo promedio de manera monótona, pasando de £4324,10 (multiplicador 0) a £7674,08 (multiplicador 1,5). Aunque los multiplicadores 0,5 y 1,0 presentan una menor desviación del costo entre folds (indicando mayor estabilidad), esta no compensa el incremento del costo promedio. El conteo de folds confirma que el multiplicador 0,5 solo mejoró el costo en 1 de los 5 folds (el fold 2, donde redujo el costo de £6713,4 a £5345,0), el multiplicador 1,0 también en 1 fold, y el multiplicador 1,5 en ninguno. Esto valida que, bajo la asimetría de costos base (£1,50 por faltante y £0,40 por exceso), añadir un colchón de seguridad incrementa sistemáticamente la penalización económica total, ya que el costo acumulado de los excedentes generados supera ampliamente el ahorro obtenido al evitar faltantes, consolidando al multiplicador 0 como la opción más eficiente en la fase de validación.


### Reto 2 · Apartado C — Seleccionar el multiplicador y evaluarlo en test
Seleccione el multiplicador con la regla común (sin mirar el test). Calcule el colchón del test con el MAE previo del entrenamiento y evalúe la política seleccionada frente a la opción sin colchón: costo, excedentes y faltantes. Compruebe si coincide con la política fijada en la sección 10.

In [ ]:
# 1. Seleccionar el multiplicador con la regla común (sin mirar el test)
r2_mult = seleccionar(r2_resumen)

# 2. Calcular el colchón del test con el MAE previo del entrenamiento
mae_t = mae_previo(SELECCIONADO, train, "reto2 · test")
colchon_test = colchon(r2_mult, mae_t)

# 3. Evaluar la política seleccionada frente a la opción sin colchón
res_sin_colchon = evaluar(test, cantidades_test[SELECCIONADO])
res_sin_colchon["opcion"] = f"{SELECCIONADO} (sin colchón)"

res_con_colchon = evaluar(test, cantidades_test[SELECCIONADO] + colchon_test)
res_con_colchon["opcion"] = f"Política seleccionada: {SELECCIONADO} + {r2_mult:g} × MAE previo"

# Crear la tabla comparativa
tabla_r2c = pd.DataFrame([res_sin_colchon, res_con_colchon]).set_index("opcion")
display(tabla_r2c.round(2))

# 4. Comprobar si coincide con la política fijada en la sección 10
print(f"Multiplicador seleccionado en validación: {r2_mult}")
print(f"MAE previo del entrenamiento: {mae_t:.1f}")
print(f"Colchón calculado para el test: {colchon_test} unidades")
print(f"¿Coincide con la política fijada en la sección 10? {r2_mult == MULT and colchon_test == COLCHON_TEST}")

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6
Política seleccionada: ExpSmoothing + 0 × MAE previo,776.62,1700.86,0.54,329.0,5884.0,8957.6


Multiplicador seleccionado en validación: 0.0
MAE previo del entrenamiento: 618.8
Colchón calculado para el test: 0 unidades
¿Coincide con la política fijada en la sección 10? True


**Interpretación 2C:**

El multiplicador seleccionado en validación fue 0,0, lo que implica un colchón de 0 unidades, dado que el MAE previo calculado exclusivamente con el historial de entrenamiento fue de 618,8 unidades. Al evaluar esta política en el conjunto de prueba, los resultados (costo de £8957,60, 329 unidades excedentes y 5884 faltantes) son idénticos a los de la opción sin colchón. Esto confirma que la política seleccionada coincide exactamente con la fijada en la sección 10, validando que, bajo la estructura de costos base (£1,50 por faltante y £0,40 por exceso), añadir un colchón de seguridad no era económicamente eficiente, ya que el costo acumulado de los excedentes generados habría superado el ahorro obtenido al mitigar los faltantes.


### Reto 2 · Apartado D — Recomendar sobre el colchón
Recomiende si corresponde añadir colchón y fundamente con la validación. Explique por qué la cantidad objetivo semanal no equivale automáticamente a una cantidad a comprar (inventario utilizable, pedidos que llegarán a tiempo y arrastre entre semanas).

**Decisión 2D:**

Con base en la evidencia de validación, no se recomienda añadir un colchón al pronóstico. Los resultados demuestran que incrementar el multiplicador (de 0 a 1,5) eleva el costo promedio de £4324,10 a £7674,08, ya que el ahorro marginal por reducir faltantes se ve ampliamente superado por el costo acumulado de las unidades excedentes generadas (que aumentan drásticamente de 30 925 a 95 926). Por lo tanto, la política óptima y más robusta es mantener un multiplicador de 0. Además, es fundamental distinguir que la cantidad objetivo semanal (pronóstico + colchón) no equivale automáticamente a la cantidad a comprar. Una orden de compra real debe descontar el inventario utilizable existente y los pedidos que llegarán a tiempo antes de la semana objetivo, y nunca puede ser negativa; consideraciones operativas que este modelo académico no simula al no disponer de datos de arrastre de inventario entre semanas.


### Reto 2 · Ampliación opcional: ventana de calibración
Repita la comparación con 1, 2 y 3 bloques de calibración y observe si cambia el multiplicador elegido. Si muestra costos de test de las políticas descartadas, identifíquelos como retrospectivos: no se usan para cambiar la política.

In [ ]:
ventanas = []

for b in [1, 2, 3]:
    # 1. Comparar políticas con 'b' bloques de calibración
    pol_folds_b = comparar_politicas(bloques=b, etiqueta=f"reto2 · validación {b} bloques")
    resumen_b = resumir(pol_folds_b, "multiplicador", MULTIPLICADORES)

    # 2. Seleccionar el multiplicador bajo esta configuración
    mult_elegido = seleccionar(resumen_b)

    # 3. Evaluar en test de forma retrospectiva (solo para observación, no cambia la política)
    mae_test_b = mae_previo(SELECCIONADO, train, f"reto2 · test {b} bloques", bloques=b)
    colchon_test_b = colchon(mult_elegido, mae_test_b)
    q_politica_test_b = cantidades_test[SELECCIONADO] + colchon_test_b
    res_test_b = evaluar(test, q_politica_test_b)

    ventanas.append({
        "bloques_calibracion": b,
        "multiplicador_elegido": mult_elegido,
        "costo_test_retrospectivo": res_test_b["costo"]
    })

df_ventanas = pd.DataFrame(ventanas).set_index("bloques_calibracion")
display(df_ventanas)

print("Nota: Los costos de test mostrados son estrictamente retrospectivos. La política fijada (multiplicador 0) no debe modificarse con base en estos resultados.")

,multiplicador_elegido,costo_test_retrospectivo
bloques_calibracion,,
1,0.0,8957.6
2,0.0,8957.6
3,0.0,8957.6


Nota: Los costos de test mostrados son estrictamente retrospectivos. La política fijada (multiplicador 0) no debe modificarse con base en estos resultados.


**Comentario opcional 2:**

La ampliación opcional confirma que, independientemente de si se utiliza 1, 2 o 3 bloques para calibrar el MAE previo, el multiplicador seleccionado se mantiene en 0.0 en todos los escenarios, generando un costo de test retrospectivo idéntico de £8957,60. Esto refuerza la robustez de la decisión inicial: bajo la estructura de costos asimétricos del caso, añadir cualquier colchón de seguridad resulta económicamente ineficiente para este SKU, ya que el costo acumulado de los excedentes supera sistemáticamente el ahorro obtenido al mitigar faltantes. Asimismo, se reitera que los costos de test mostrados son estrictamente retrospectivos y no deben utilizarse para modificar la política fijada antes de la evaluación.


## Reto 3 — ¿Funciona la misma solución para otro producto?

**Caso.** La empresa quiere extender el procedimiento a un segundo producto de alta rotación antes de aplicarlo a toda la cartera. La gerencia pregunta si el método que se eligió para el primer producto puede generalizarse.

**Problema.** ¿Qué modelo se selecciona para el segundo producto con las mismas reglas, cómo le fue en test frente a Naïve y qué haría la empresa con cada producto?

**Datos, supuestos y recursos**
- Segundo SKU de la lista de elegibles, definida solo con el entrenamiento (no se elige mirando el test).
- Mismas semanas completas, limpieza, backtesting, costos base y regla común de selección.
- Se comparan modelos **sin colchón** para mantener el alcance claro; la política de colchón se trabajó en el Reto 2.

**Apartados**
- **A.** Identificar el producto y describir su serie
- **B.** Backtesting y selección con la regla común
- **C.** Evaluar en test y comparar ambos productos
- **D.** Recomendación consolidada

**Evidencias que deben quedar en el notebook**
- Serie del segundo producto y tabla descriptiva del entrenamiento frente al primero, con su gráfico.
- Tabla de backtesting por modelo y seleccionado.
- Tabla comparativa de ambos productos con modelo, costos, ahorro o sobrecosto y resultado.
- Interpretaciones A–D.

### Reto 3 · Apartado A — Identificar el producto y describir su serie
Identifique el segundo SKU elegible, construya su serie con las mismas semanas completas y separe entrenamiento y test. Describa su comportamiento solo con el entrenamiento y compárelo con el primer producto (tabla y gráfico).

In [ ]:
assert len(elegibles) >= 2, "Se necesita un segundo SKU elegible"
SKU2 = elegibles.loc[1, "StockCode"]
desc2 = uk.loc[uk["StockCode"] == SKU2, "Description"].dropna().astype(str).mode().iloc[0]

# Construcción de la serie del segundo producto con las mismas semanas completas
serie2 = (uk[uk["StockCode"] == SKU2].groupby("week")["Quantity"].sum()
          .reindex(semanas, fill_value=0).astype(float))
serie2.name = "unidades"
train2 = serie2.loc[semanas_train]
test2 = serie2.loc[semanas_test]

# Tabla descriptiva comparativa del entrenamiento
desc_comparativa = pd.DataFrame({
    "SKU": [SKU, SKU2],
    "Descripción": [desc, desc2],
    "Unidades totales (train)": [int(train.sum()), int(train2.sum())],
    "Semanas con venta": [int((train > 0).sum()), int((train2 > 0).sum())],
    "Media semanal": [round(train.mean(), 1), round(train2.mean(), 1)],
    "Mediana semanal": [round(train.median(), 1), round(train2.median(), 1)],
    "Máximo semanal": [int(train.max()), int(train2.max())]
})
display(desc_comparativa)

# Gráfico de series de entrenamiento
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, mode="lines", name=f"SKU 1: {SKU}"))
fig.add_trace(go.Scatter(x=train2.index, y=train2.values, mode="lines", name=f"SKU 2: {SKU2}"))
fig.update_layout(
    title="Comparación de series de entrenamiento (SKU 1 vs SKU 2)",
    xaxis_title="Semana (lunes)",
    yaxis_title="Unidades vendidas",
    hovermode="x unified"
)
fig.show()

,SKU,Descripción,Unidades totales (train),Semanas con venta,Media semanal,Mediana semanal,Máximo semanal
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,88273,93,919.5,584.5,7063
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,80903,94,842.7,718.5,3329


**Interpretación 3A:**

La tabla comparativa y el gráfico evidencian que, si bien ambos SKUs cumplen los criterios de elegibilidad y presentan volúmenes totales similares (88 273 vs 80 903 unidades), exhiben patrones de demanda marcadamente distintos. El SKU 2 (85123A) muestra un comportamiento más estable y predecible: su mediana semanal es significativamente mayor (718,5 vs 584,5 unidades) y su máximo (3 329) es menos de la mitad del registrado por el SKU 1 (7 063), lo que indica una menor volatilidad y picos menos extremos. En contraste, el SKU 1 presenta una media elevada (919,5) distorsionada por eventos atípicos, mientras que su mediana revela un nivel base más bajo. Esta diferencia es relevante para la generalización del método, ya que un modelo que funciona bien para un producto volátil con picos extremos podría no ser óptimo para uno más estable, y viceversa, lo que justifica evaluar el desempeño de los modelos en el segundo producto antes de extender la solución a toda la cartera.


### Reto 3 · Apartado B — Backtesting y selección con la regla común
Ejecute el mismo backtesting con Naïve, Naïve estacional y Exponential Smoothing (`backtest`), resuma los resultados y seleccione con la regla común. Presente la tabla e indique si la selección se decidió por la tolerancia del 1 %.

In [ ]:
# Ejecutar backtesting para el segundo producto
filas2, q_bt2 = backtest(train2, MODELOS, "reto3")

# Resumir resultados y seleccionar el modelo con la regla común
tabla2 = resumir(filas2, "modelo", MODELOS)
SELECCIONADO2 = seleccionar(tabla2)

display(tabla2.round(2))

# Verificación automática de si la tolerancia del 1% fue el factor decisivo
costo_min = tabla2["costo_prom"].min()
costo_sel = tabla2.loc[SELECCIONADO2, "costo_prom"]

if abs(costo_sel - costo_min) < 1e-9:
    print(f"El modelo seleccionado es '{SELECCIONADO2}', que presentó el costo promedio mínimo absoluto (£{costo_sel:.2f}). La tolerancia del 1% confirmó su elegibilidad, y la selección se definió por ser el mínimo directo o por ganar los desempates (MAE, desviación o complejidad).")
else:
    print(f"El modelo seleccionado es '{SELECCIONADO2}'. Aunque no fue el de costo absoluto mínimo, su costo (£{costo_sel:.2f}) se mantuvo dentro de la tolerancia del 1% del mínimo (£{costo_min:.2f}), y resultó elegido al ganar los criterios de desempate (menor MAE, menor desviación del costo o menor complejidad).")

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,4814.82,3483.34,670.62,656.73,867.66,0.97,14694.0,12131.0,0
SeasonalNaive,3306.84,2147.86,459.32,290.28,665.50,0.67,10023.0,8350.0,1
ExpSmoothing,2591.80,1907.11,475.12,266.42,569.80,0.71,14135.0,4870.0,2


El modelo seleccionado es 'ExpSmoothing', que presentó el costo promedio mínimo absoluto (£2591.80). La tolerancia del 1% confirmó su elegibilidad, y la selección se definió por ser el mínimo directo o por ganar los desempates (MAE, desviación o complejidad).


**Interpretación 3B:**

El backtesting del segundo producto (SKU 85123A) confirma que ExpSmoothing fue el modelo seleccionado, al presentar el menor costo promedio de validación absoluto (£2591,80), por lo que la elección se definió directamente por este mínimo sin necesidad de recurrir a la tolerancia del 1 % o a los criterios de desempate. Es fundamental destacar que, aunque SeasonalNaive obtuvo el menor error estadístico (MAE de 459,32 frente a 475,12 de ExpSmoothing), no resultó ser la opción más económica. Esto valida el principio central del laboratorio: la asimetría de costos hace que la reducción de unidades faltantes lograda por ExpSmoothing (4870 frente a 8350 de SeasonalNaive) compense ampliamente su ligeramente mayor error absoluto, demostrando que el menor error estadístico no garantiza el menor impacto financiero.


### Reto 3 · Apartado C — Evaluar en test y comparar ambos productos
Evalúe en test el modelo seleccionado y Naïve para el segundo producto. Construya una tabla comparativa de ambos productos (sin colchón) con modelo seleccionado, costo de validación, costo de test del seleccionado y de Naïve, ahorro o sobrecosto, folds en que superó a Naïve y resultado de la evaluación, aplicando la regla de adopción de la sección 13.

In [ ]:
# 1. Costos de test para el segundo producto (sin colchón)
cant_test2 = {m: pronosticar(m, train2, TEST_H, "test r3") for m in MODELOS}
res_test2 = pd.DataFrame({m: evaluar(test2, cant_test2[m]) for m in MODELOS}).T

# 2. Folds ganados para el producto 1 (comparando modelo seleccionado vs Naive en backtest)
bt_sel_1 = bt[bt["modelo"] == SELECCIONADO].set_index("fold")["costo"]
bt_naive_1 = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]
folds_ganados_1 = int((bt_sel_1 < bt_naive_1).sum())

# 3. Folds ganados para el producto 2 (usando filas2 del Apartado B)
bt2 = pd.DataFrame(filas2)
bt_sel_2 = bt2[bt2["modelo"] == SELECCIONADO2].set_index("fold")["costo"]
bt_naive_2 = bt2[bt2["modelo"] == "Naive"].set_index("fold")["costo"]
folds_ganados_2 = int((bt_sel_2 < bt_naive_2).sum())

# 4. Función auxiliar para aplicar la regla de adopción de la sección 13
def evaluar_decision(costo_sel, costo_naive, folds_ganados, total_folds):
    if costo_sel < costo_naive and folds_ganados > total_folds / 2:
        return "Adoptar como piloto"
    elif costo_sel < costo_naive:
        return "Mantener en evaluación"
    else:
        return "No adoptar"

# 5. Construcción de la tabla comparativa
comparacion = pd.DataFrame({
    "Producto": [f"SKU 1: {SKU}", f"SKU 2: {SKU2}"],
    "Modelo seleccionado": [SELECCIONADO, SELECCIONADO2],
    "Costo validación (£)": [
        round(resumen_val.loc[SELECCIONADO, "costo_prom"], 2),
        round(tabla2.loc[SELECCIONADO2, "costo_prom"], 2)
    ],
    "Costo test seleccionado (£)": [
        round(res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"], 2),
        round(res_test2.loc[SELECCIONADO2, "costo"], 2)
    ],
    "Costo test Naïve (£)": [
        round(res_test.loc["Naive (sin colchón)", "costo"], 2),
        round(res_test2.loc["Naive", "costo"], 2)
    ],
    "Ahorro/Sobrecosto (£)": [
        round(res_test.loc["Naive (sin colchón)", "costo"] - res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"], 2),
        round(res_test2.loc["Naive", "costo"] - res_test2.loc[SELECCIONADO2, "costo"], 2)
    ],
    "Folds superados a Naïve": [f"{folds_ganados_1} de {len(origenes)}", f"{folds_ganados_2} de {len(origenes)}"],
    "Decisión": [
        evaluar_decision(
            res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"],
            res_test.loc["Naive (sin colchón)", "costo"],
            folds_ganados_1, len(origenes)
        ),
        evaluar_decision(
            res_test2.loc[SELECCIONADO2, "costo"],
            res_test2.loc["Naive", "costo"],
            folds_ganados_2, len(origenes)
        )
    ]
}).set_index("Producto")

display(comparacion)

,Modelo seleccionado,Costo validación (£),Costo test seleccionado (£),Costo test Naïve (£),Ahorro/Sobrecosto (£),Folds superados a Naïve,Decisión
Producto,,,,,,,
SKU 1: 84077,ExpSmoothing,4324.1,8957.6,7087.9,-1869.7,4 de 5,No adoptar
SKU 2: 85123A,ExpSmoothing,2591.8,3432.0,6640.7,3208.7,5 de 5,Adoptar como piloto


**Interpretación 3C:**

La tabla comparativa evidencia que, aunque el modelo ExpSmoothing fue seleccionado para ambos productos durante la validación, su desempeño en el conjunto de prueba divergió radicalmente, invalidando la generalización automática del método a toda la cartera. Para el SKU 1 (84077), a pesar de superar a Naïve en 4 de 5 folds, incurrió en un sobrecosto del 26,4 % (£1869,70) en el test debido al pico atípico no modelable, consolidando la decisión de "No adoptar". En contraste, para el SKU 2 (85123A), el mismo modelo demostró una robustez excepcional, superando a Naïve en los 5 folds de validación y generando un ahorro sustancial de £3208,70 en el test. Esta divergencia confirma que la decisión de adopción debe ser específica para cada producto, ya que la estabilidad inherente de la demanda del SKU 2 permitió que el suavizamiento exponencial capturara su patrón eficazmente, a diferencia de la alta volatilidad del SKU 1.


### Reto 3 · Apartado D — Recomendación consolidada
Prepare una recomendación para la empresa: qué haría con cada producto (adoptar, mantener en evaluación o no adoptar), qué evidencia la respalda y por qué dos productos no permiten generalizar el método a toda la cartera.

**Decisión 3D:**

Con base en la evidencia, la recomendación es no adoptar ExpSmoothing como sustituto de Naïve para el SKU 1 (84077), dado que incurrió en un sobrecosto del 26,4 % en el test debido a un pico de demanda atípico, por lo que Naïve debe mantenerse como referencia operativa. Por el contrario, para el SKU 2 (85123A), se recomienda adoptar ExpSmoothing como piloto controlado, ya que generó un ahorro sustancial de £3208,70 frente a Naïve, demostrando que su mayor complejidad sí agrega valor económico para series más estables. Sin embargo, estos resultados no permiten generalizar el método a toda la cartera, ya que el desempeño óptimo depende intrínsecamente del patrón de demanda de cada producto; mientras que un modelo complejo puede mitigar eficazmente los faltantes en series regulares (SKU 2), resulta vulnerable a eventos atípicos en series volátiles (SKU 1), por lo que la decisión de adopción debe validarse individualmente para cada SKU antes de su implementación masiva.


### Revisión de advertencias y verificaciones automáticas

Se resumen las advertencias de **todos** los ajustes de Exponential Smoothing (backtesting, calibración, test, reentrenamiento final y retos) y se comprueban las condiciones de validez del flujo.

In [ ]:
aj = pd.DataFrame(registro_ajustes)
display(aj.groupby("etapa").agg(ajustes=("ajuste", "count"), con_advertencias=("advertencias", lambda s: int((s > 0).sum())),
                                tipos=("tipos", lambda s: "; ".join(sorted({t for x in s for t in x.split("; ") if t}))))
          .sort_values("ajustes", ascending=False))
mensajes = sorted({m for x in aj["mensajes"] for m in x.split(" | ") if m})
print("Mensajes distintos:", mensajes if mensajes else "ninguno")
print("Especificaciones usadas:", aj["especificacion"].value_counts().to_dict())

verificaciones = {
    "Ningún fold invade el test": all(train.index[o + BT_H - 1] < semanas_test[0] for o in origenes),
    "Colchón de cada fold calibrado solo con semanas anteriores a su origen": bool((pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all()),
    "Colchón del test calculado solo con el entrenamiento": len(train) == len(semanas_train) and train.index[-1] < semanas_test[0],
    "La tabla final usa el multiplicador fijado": COLCHON_FINAL == colchon(MULT, MAE_FINAL),
    "El costo de la política en test se recalcula con su colchón": np.isclose(costo_pol, evaluar(test, q_politica)["costo"]),
}
for k, v in verificaciones.items():
    print(("OK   " if v else "FALLA"), k)

,ajustes,con_advertencias,tipos
etapa,,,
reto2,58,0,
calibración,10,0,
backtesting,5,0,
reto3,5,0,
calibración final,2,0,
calibración test,2,0,
final,1,0,
test,1,0,
test r3,1,0,


Mensajes distintos: ninguno
Especificaciones usadas: {'nivel + tendencia + estacionalidad aditivas (m = 4)': 85}
OK    Ningún fold invade el test
OK    Colchón de cada fold calibrado solo con semanas anteriores a su origen
OK    Colchón del test calculado solo con el entrenamiento
OK    La tabla final usa el multiplicador fijado
OK    El costo de la política en test se recalcula con su colchón


### Retroalimentación final

**Pregunta 10.** Indique dos razones por las que el modelo podría fallar en producción aun cuando el backtesting haya sido bueno.

**Respuesta:**

El modelo podría fallar en producción por dos razones principales: primero, la ocurrencia de eventos atípicos no modelables, como el pico extraordinario de 5 328 unidades en la semana del 24 de octubre de 2011 (donde un solo pedido concentró el 90,1 % del volumen), que los modelos univariantes no pueden anticipar al basarse exclusivamente en patrones históricos de demanda. Segundo, cambios estructurales en el patrón de demanda o la influencia de variables exógenas no consideradas, como promociones comerciales, campañas de marketing, cambios de precio, acciones de la competencia o factores estacionales no capturados en el periodo de entrenamiento, los cuales pueden alterar sustancialmente el comportamiento futuro de la serie sin que el modelo tenga información para ajustarse a tiempo.


# 15. Ticket de salida

«El modelo seleccionado antes del test fue ExpSmoothing, con la política ExpSmoothing + 0 × MAE previo (costo promedio de validación £4324.10). En el test costó £8957.60 frente a £7087.90 de Naïve. Mi decisión es no adoptar porque generó un sobrecosto del 26,4 % debido a un pico de demanda atípico no anticipado, sin lograr superar al baseline en el periodo de prueba. La proyección académica de 9291 unidades no es una orden de compra porque no descuenta el inventario utilizable existente ni los pedidos en tránsito, ni simula el arrastre de stock entre semanas.»


# 16. Conclusiones

Registre tres conclusiones de carácter técnico y de negocio.

1. La optimización del costo económico prevalece sobre la precisión estadística pura. La selección de modelos basada en costos asimétricos (donde el faltante penaliza 3,75 veces más que el exceso) demostró que un modelo con mayor error absoluto (MAE/RMSE), como Exponential Smoothing, puede ser la opción más rentable en validación al mitigar eficazmente las unidades faltantes, alineando así el pronóstico con los objetivos financieros y de nivel de servicio del negocio.

2. La vulnerabilidad inherente de los modelos univariantes ante eventos atípicos no modelables. El sobrecosto observado en el test del SKU 1 no se debió a una falla estructural del algoritmo, sino a un pico de demanda extraordinario (un solo pedido que concentró el 90,1 % del volumen semanal) que ningún método basado únicamente en historia pasada puede anticipar, evidenciando la necesidad de incorporar variables exógenas o reglas de negocio para gestionar promociones y pedidos masivos en la vida real.

3. La imposibilidad de generalizar una única solución a toda la cartera de productos. La evaluación comparativa demostró que, mientras Exponential Smoothing generó un sobrecosto del 26,4 % en el SKU 1 (alta volatilidad), produjo un ahorro sustancial (£3208,70) en el SKU 2 (demanda más estable y predecible). Esto valida que la estrategia de abastecimiento debe segmentarse y validarse individualmente por SKU, evitando la implementación masiva de un modelo sin evaluar previamente su compatibilidad con el patrón de demanda específico de cada producto.

# 17. Evidencia de entrega

- [ ] Fuente, hojas, dimensiones, rango de fechas y versiones visibles.
- [ ] Limpieza documentada (incluidos `StockCode` nulos).
- [ ] Semanas de lunes a domingo; semanas incompletas excluidas.
- [ ] Test de 8 semanas reservado antes de seleccionar el SKU, el modelo y la política.
- [ ] SKU seleccionado con criterios previos al test y límites del criterio de disponibilidad explicados.
- [ ] Exploración solo con entrenamiento.
- [ ] Backtesting de 8 semanas con fechas de cada fold y comprobación de que no invade el test.
- [ ] Regla única de selección aplicada a modelo, política y retos.
- [ ] Colchón calibrado solo con errores previos a cada origen.
- [ ] Política fijada antes del test y evaluada una vez.
- [ ] Decisión de adopción separada de la proyección académica.
- [ ] Tres retos individuales con sus apartados A, B, C y D resueltos e interpretados; advertencias revisadas.
- [ ] Notebook en GitHub (`semana10/`) y en su propio Colab; ambos URLs presentados en la actividad de la semana.
- [ ] Uso de IA declarado.

# Referencias
- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Sharda, R., Delen, D., & Turban, E. *Business Intelligence, Analytics, Data Science and AI*. Pearson.